# RQ2/RQ4 checkpoint completion + RQ1 two-way interactions (self-contained Kaggle driver)

Implements [`docs/TASK_RQ2_COMPLETION_AND_RQ1_INTERACTIONS.md`](../docs/TASK_RQ2_COMPLETION_AND_RQ1_INTERACTIONS.md)
end to end. **Thesis numbering throughout.** In the code, `rq1_verdict` = **RQ2** (objective vs
score) and `rq2_table` / `rq2_verdict` = **RQ4** (post-hoc refit).

| Part | What it does | Compute |
|---|---|---|
| **B** (runs first) | RQ1's ten two-way interaction terms over the per-seed grid table, with a seed bootstrap | CPU, ~10 s |
| **A** | Train and score the 21 CIFAR-FS 5-shot cells whose checkpoints were never recovered, check each against the committed grid, then re-aggregate RQ2 and RQ4 | GPU, **~10 h measured** |

Part B runs first because it needs no checkpoint, so its tables exist even if the long GPU part dies.

**Kaggle settings:** Accelerator **GPU T4** (the Step 10 grid ran on a T4, and the regression
guard compares against those numbers), Internet **ON**. Attach:
- `beft-thesis-data` (owner `notavailable73`): CIFAR-100 / SVHN / TinyImageNet. Anything
  missing downloads at runtime.
- **From session 2 on:** this notebook's own outputs from the previous session
  (`rq-completion-results` + `rq-completion-checkpoints`, or the previous version's notebook
  output). Section 5 restores them and the run resumes where it stopped.
- Optional: any Step 10 artifact dataset. Recovery (task A.1) scans it; see fact 1 below.

Run with **Save Version → Save & Run All (Commit)** so closing the tab does not kill a 9 h run.

### Checked before shipping (against the committed repo, 2026-09-14)

1. **The 21 checkpoints are not in the Step 10 artifacts.** `step10a_cifar_5shot_artifacts.zip`
   holds 15 checkpoints for this slice (mbnet/lora/softmax ×3, full_ft ×6, linear_probe ×6), and all
   15 belong to the 99 cells already scored. Recovery still runs, but plan on retraining.
2. **Retraining is ~10 GPU-hours, not 5–7.** Summing the measured Step 10 wall times of these 21
   configs (`results/grid/_run_log.jsonl`) gives 607 min. One Kaggle session cannot hold that
   safely, so the run is budgeted (`MAX_MINUTES`) and resumable. **Expect two sessions.**
3. **One committed baseline is a smoke-test result.**
   `results/grid/grid_cifar_5shot_mbnet_lora_seed42_lora_prototype-evidential_metrics.json` has
   `num_episodes: 20` and no TinyImageNet/Gaussian keys. It is Step 10a's pre-flight smoke test
   (run log line 2). The real grid run logged `skipped_done` for that cell (line 22) because
   `--resume` found the JSON. Consequences:
   - The 600-episode regression guard for that cell will read `MISMATCH` whatever happens.
     Section 8 re-scores the checkpoint on the same first 20 test seeds, and **that** guard is
     the one that counts.
   - It is the only one of the 120 grid JSONs like this, and the source of RQ1's "95/96
     TinyImageNet" gap. The accuracy/ECE/SVHN values `mvt_results.json` holds for that
     observation are 20-episode estimates too. Section 10 reruns Part B with that one
     observation replaced by the same model's 600-episode scores, but only if the 20-episode
     guard passes.
4. **The metric keys behind RQ1's published table.** All four rows reproduce to 2 d.p., and
   Section 4 re-checks them. Accuracy = `accuracy_mean`. **ECE = `ece_pooled`**, not
   `ece_per_episode_mean` as the task doc guessed (that key gives head 83.03%, residual 10.55%).
   Far-OOD = `ood_auroc__svhn_far__<native>` alone. Near = `ood_auroc__tin_near__<native>`.
   Native score = vacuity for evidential, msp for softmax.
5. **The published TinyImageNet row is an OLS Type-II fit, not `eta_squared`.** On those 95
   unbalanced rows, `eta_squared` gives 42.92 / 22.90 / 21.02 (residual 10.67) where the table
   says 42.60 / 21.98 / 20.98 (11.42). Type II reproduces the published numbers, and on balanced
   rows the two methods agree to 1e-14. The balanced outcomes use `eta_squared`, as the task asks.
   The one unbalanced outcome uses Type II, and both numbers are saved.
6. **The "163× / 394×" instability is this gap.** On the committed 99 records, RQ2's far-OOD
   score/objective ratio is 163.4× over all records and 394.4× once the half-present
   `cifar_fs/5shot/mobilenetv3_small/lora` design is dropped. `eta_squared` reports
   `balanced: True` both times, because it cannot see a missing arm. Section 9 adds that check.

**Self-contained.** The only new logic is `scripts/rq_completion.py`, written out by Section 3.
Training, scoring, the evidence map (`PrototypeHead.to_evidence`), the regression guard and the η²
decomposition all come unchanged from the clone (`scripts/train.py`, `rq_core`, `rq_drivers`,
`rq_aggregate`). Nothing writes to `configs/`, `results/grid/`, `results/mvt_results.json`,
`results/rq_summary.json` or `results/rq_checkpoint_audit.json`, and Section 11 checks that with
`git status`.

## 0. Control panel (the only cell you normally edit)

Everything is **resumable**: a cell whose `results/rq_factorial/<cell>.json` exists is skipped.
`MAX_MINUTES` stops new cells from starting; the one in flight always finishes. Part B and the
aggregation are cheap and recompute every session from whatever is on disk.

In [1]:
# ---- What to run this session -------------------------------------------
RUN_PART_B     = True   # §4      RQ1 interactions from the committed grid (CPU, ~10 s)
RUN_PART_A     = True   # §5,7,8  restore, train + score the 21 cells, regression guards (GPU)
RUN_SMOKE_EVAL = True   # §6      3-episode data/pool check on an untrained model (~3 min)
RUN_AGGREGATE  = True   # §9-10   RQ2/RQ4 re-aggregation + RQ1 sensitivity (CPU; safe on a partial run)

# ---- Part A scope --------------------------------------------------------
# Filter keys: backbone, adapter, head, seed. Empty = all 21.
#   'backbone': 'resnet18' | 'mobilenetv3_small'
#   'adapter' : 'bottleneck_parallel' | 'lora'
# Leave empty. The run order finishes a whole design (both heads x 3 seeds) before starting the
# next, beginning with mobilenetv3_small/lora, the half-present design behind RQ2's instability.
ONLY = {}

NUM_EPISODES = 600      # frozen test seeds 0..599. Any other value writes to a scratch dir, never rq_factorial/
MAX_MINUTES  = 540.0    # no new cell starts after this. The cell in flight can take ~45 min more;
                        # 540 + 45 + setup + packing stays inside Kaggle's 12 h commit limit.
RECOVER_SEARCH_ROOTS = ('/kaggle/input',)

# ---- Part B --------------------------------------------------------------
N_BOOT    = 2000        # seed-bootstrap resamples (task B.3); 0 disables
BOOT_SEED = 20260914    # fixed, so the intervals are reproducible

# ---- Misc ---------------------------------------------------------------
PERSIST_LOGITS   = True    # ~12 MB/cell into results/rq_logits/, as Phase A did: future re-scoring is free
PACK_CHECKPOINTS = True    # second zip (~600 MB). Lost checkpoints caused this task, so keep them
WANDB_MODE       = 'disabled'
CACHE_OOD_IMAGES = True    # ~1.2 GB host RAM; loads the OOD pools once per session, not once per cell

print('parts:', {'B': RUN_PART_B, 'A': RUN_PART_A, 'smoke': RUN_SMOKE_EVAL,
                 'aggregate': RUN_AGGREGATE}, '| filter:', ONLY,
      '| episodes:', NUM_EPISODES, '| budget:', MAX_MINUTES, 'min')

parts: {'B': True, 'A': True, 'smoke': True, 'aggregate': True} | filter: {} | episodes: 600 | budget: 540.0 min


## 1. GPU check + clone repo + install deps

In [2]:
import os, subprocess, sys, time
# cuBLAS reads this when its handle is first created, which in this notebook happens in-process.
# set_seed() sets the same value, but only via setdefault, so pin it before any CUDA maths runs.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch

print('python:', sys.version.split()[0], '| torch:', torch.__version__, '| cuda:', torch.version.cuda)
print('gpu   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('WARNING: no GPU (Settings > Accelerator > GPU T4). Part B and the aggregation run fine '
          'on CPU; training the 21 cells does not.')
elif 'T4' not in torch.cuda.get_device_name(0):
    print('NOTE: not a T4. The committed grid was produced on a T4, so a regression-guard '
          'difference on this GPU may be hardware, not a defect. It is still recorded as found.')

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/thesis' if os.path.isdir('/kaggle/working') else os.path.abspath('./thesis')

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

# The repo root for `src.*` / `scripts.*`, and scripts/ itself because rq_core / rq_drivers /
# rq_aggregate / rq_completion import each other by bare name.
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))

from pathlib import Path
REPO = Path(os.getcwd())
REPO_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                           capture_output=True, text=True).stdout.strip()
print('repo ready at', REPO, '| HEAD', REPO_HEAD)

python: 3.12.13 | torch: 2.10.0+cu128 | cuda: 12.8
gpu   : Tesla T4


Cloning into '/kaggle/working/thesis'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 5.6 MB/s eta 0:00:00
repo ready at /kaggle/working/thesis | HEAD 8d706ea


## 2. Stage data (symlink the attached Kaggle dataset into `data/`)

The 21 cells are all CIFAR-FS, so this session needs CIFAR-100 (ID + near-OOD), SVHN and
TinyImageNet. The staged-path finders are the same ones `scripts/train.py` / `evaluate.py` call
at runtime. The grid configs are committed, so `build_grid_configs.py` is deliberately **not** run:
nothing in `configs/` gets regenerated, and Section 11 verifies that.

In [3]:
import shutil

from src.datasets.cifar_fs import _find_staged_cifar100_root
from src.datasets.svhn_ood import _find_staged_svhn_root
from src.datasets.tinyimagenet_ood import _find_extracted_tin_root

LINKS = {}
if (root := _find_staged_cifar100_root('data')):
    LINKS['data/cifar-100-python'] = os.path.join(root, 'cifar-100-python')
if (root := _find_staged_svhn_root('data')):
    LINKS['data/svhn/test_32x32.mat'] = os.path.join(root, 'test_32x32.mat')
if (root := _find_extracted_tin_root('data')):
    LINKS['data/tiny-imagenet-200'] = root

os.makedirs('data/svhn', exist_ok=True)
if not LINKS:
    print('No staged files under /kaggle/input -- did you attach `beft-thesis-data`? '
          'Falling back to runtime downloads for everything.')
for link, target in LINKS.items():
    if os.path.exists(link) or os.path.islink(link):
        print(f'OK   (already present): {link}')
        continue
    if not os.path.exists(target):
        print(f'MISSING source -- {os.path.basename(link)} will download at runtime')
        continue
    try:
        os.symlink(target, link)
        print(f'OK   (symlinked): {link}')
    except OSError as e:
        print(f'symlink failed ({e}); copying instead: {link}')
        (shutil.copytree if os.path.isdir(target) else shutil.copy2)(target, link)

# Frozen splits (idempotent; re-materialises the canonical files data/ needs after a fresh clone).
subprocess.run([sys.executable, 'scripts/build_cifar_fs_split.py'], check=True)
subprocess.run([sys.executable, 'scripts/build_mini_imagenet_split.py'], check=True)
print('\nsplits ready')

OK   (symlinked): data/cifar-100-python
OK   (symlinked): data/svhn/test_32x32.mat
OK   (symlinked): data/tiny-imagenet-200
wrote /kaggle/working/thesis/data/cifar_fs_split.json  (64/16/20, disjoint, union=100, status=canonical_bertinetto_via_torchmeta)
wrote /kaggle/working/thesis/data/mini_imagenet_split.json  (64/16/20, disjoint, union=100, status=canonical_ravi_larochelle)

splits ready


## 3. The new code: `scripts/rq_completion.py`

One module, written into the clone so the source lives in this `.ipynb` and travels back in the
results zip. The module docstring explains its three new pieces:

| Piece | Task step | Why it exists |
|---|---|---|
| `select_cells`, `preflight`, `restore_session_artifacts`, `coverage`, `record_provenance` | A.1–A.2 | Pick the 21 cells (cross-checked against `rq_checkpoint_audit.json`), recover or resume, estimate from measured wall times |
| `smoke_baseline_guards`, `guard_table` | A.4 | The first-20-episode guard for the smoke baseline, plus `best_val_epoch` / `n_params` checks that do not depend on evaluation |
| `rq2_completeness`, `rq2_rq4_aggregate` | A.5 | Catch a wholly missing arm, which `balanced` cannot see; confirm the unchanged aggregator reproduces the committed summary before anything new is quoted |
| `flatten`, `decompose_outcome`, `ols_type2`, `seed_bootstrap` | B.1–B.3 | The per-seed table, `eta_squared` with interactions, the Type-II cross-check, the bootstrap |

In [4]:
%%writefile scripts/rq_completion.py
"""RQ2/RQ4 checkpoint completion + RQ1 two-way interactions.

Implements docs/TASK_RQ2_COMPLETION_AND_RQ1_INTERACTIONS.md. The source lives
in notebooks/rq2_completion_rq1_interactions.ipynb, which writes it to
scripts/rq_completion.py in the clone.

Part A  The 21 CIFAR-FS 5-shot grid cells whose checkpoints were never
        recovered are trained and scored by the EXISTING Phase A driver
        (`rq_drivers.run_phase_a(allow_retrain=True)`). RQ2 and RQ4 are then
        re-aggregated by the EXISTING `rq_aggregate` functions.
Part B  RQ1's two-way interaction terms, from the EXISTING
        `rq_aggregate.eta_squared` (which always computed them) over the
        per-seed table in results/mvt_results.json.

Retired-numbering warning: `rq1_verdict` in code = RQ2 in the thesis/deck
(objective vs score); `rq2_table`/`rq2_verdict` in code = RQ4 (post-hoc
refit). Every label this module PRINTS uses the thesis numbering.

No model is scored, no evidence mapped and no variance decomposed here by new
code, with three deliberate exceptions:

1. Smoke-baseline guard (task A.4). The committed baseline for
   cifar_5shot_mbnet_lora_evidential_seed42 is NOT a 600-episode evaluation:
   its results/grid JSON has num_episodes=20 and no tin_near/gaussian_far keys.
   It is the Step 10a pre-flight smoke test (`--num-episodes 20`, no OOD
   extras). results/grid/_run_log.jsonl shows it `ok` at line 2 and the real
   grid run `skipped_done` at line 22, because `--resume` found that JSON. It
   is the only one of the 120 committed grid JSONs like this, and it is where
   RQ1's "95/96 TinyImageNet observations" gap comes from. A 600-episode
   regression guard against it reads MISMATCH whatever happens, so the check
   that means something is the same checkpoint re-scored on the same first 20
   test seeds. `smoke_baseline_guards` does that (training does not depend on
   --num-episodes; evaluate.py and factorial_run_one both take seeds[:n]).

2. RQ2 completeness check (task A.5.2). eta_squared's `balanced` flag compares
   the counts of cells that are PRESENT, so a (design, objective) arm with zero
   observations is invisible to it. That is exactly the 99-cell state:
   cifar_fs/5shot/mobilenetv3_small/lora had softmax records and no
   evidential ones. `rq2_completeness` enumerates the full crossing instead.

3. OLS Type-II decomposition (Part B). The published TinyImageNet-near RQ1 row
   (1.64 / 42.60 / 0.61 / 21.98 / 20.98, residual 11.42) is a Type-II
   main-effects fit on the 95 unbalanced rows. eta_squared gives
   1.79 / 42.92 / 0.70 / 22.90 / 21.02, residual 10.67 on the same rows; its
   docstring says it is exact only when the design is balanced. With +-1 coding
   on a balanced 2-level design the two methods agree exactly, and
   `decompose_outcome` checks that on the three balanced outcomes before any
   OLS number is used. OLS also turns the seed bootstrap (task B.3) into one
   matrix product per model instead of thousands of Python loops.
"""
from __future__ import annotations

import json
import math
import os
import shutil
import subprocess
import time
import zipfile
from itertools import combinations
from pathlib import Path

import numpy as np

import rq_aggregate as A

OUT_SUBDIR = "rq_completion"
EXPECTED_N_EPISODES = 600

# =====================================================================
# Part A: which cells
# =====================================================================
#: Verbatim from the task doc, and cross-checked against
#: results/rq_checkpoint_audit.json["missing_cells"] in `select_cells`.
MISSING_CONFIGS = frozenset(
    f"configs/grid/cifar_5shot_{bb}_{ad}_{head}_seed{seed}.yaml"
    for bb, ad, heads in (("r18", "parallel", ("evidential", "softmax")),
                          ("r18", "lora", ("evidential", "softmax")),
                          ("mbnet", "parallel", ("evidential", "softmax")),
                          ("mbnet", "lora", ("evidential",)))
    for head in heads
    for seed in (42, 43, 44)
)

#: Committed grid baselines known NOT to be 600-episode evaluations.
EXPECTED_SMOKE_BASELINES = frozenset(
    {"configs/grid/cifar_5shot_mbnet_lora_evidential_seed42.yaml"})

#: Run order. Each design (backbone x adapter) is finished, both heads x 3
#: seeds, before the next one starts. A session that stops on its time budget
#: then never leaves a design with only one objective arm, which is the
#: half-present state that made the 99-cell RQ2 ratio unstable.
#: mobilenetv3_small/lora goes first because it is the one design that is
#: half-present TODAY (softmax recovered, evidential not). Its 3 cells alone
#: restore RQ2's objective x score crossing. The rest follow in increasing
#: Step 10 wall time.
DESIGN_ORDER = (("mobilenetv3_small", "lora"),
                ("resnet18", "lora"),
                ("mobilenetv3_small", "bottleneck_parallel"),
                ("resnet18", "bottleneck_parallel"))


def cell_id(c: dict) -> str:
    """Identical to the id rq_drivers.run_phase_a gives each output JSON."""
    return (f"{c['dataset']}_{c['k_shot']}shot_{c['backbone']}_"
            f"{c['adapter']}_{c['head']}_seed{c['seed']}")


def design_of(c: dict) -> str:
    """Identical to rq_aggregate.design_key for a grid-index cell."""
    return f"{c['dataset']}/{c['k_shot']}shot/{c['backbone']}/{c['adapter']}"


def grid_index(repo_root: Path) -> list[dict]:
    return json.load(open(repo_root / "configs" / "grid" / "_index.json"))["cells"]


def select_cells(repo_root: Path, only: dict | None = None) -> list[dict]:
    """The 21 cells, in run order, optionally filtered by `only`."""
    cells = [c for c in grid_index(repo_root) if c["config"] in MISSING_CONFIGS]
    if len(cells) != 21:
        raise RuntimeError(f"expected 21 cells in configs/grid/_index.json, got "
                           f"{len(cells)} -- re-check the audit list")
    audit = json.load(open(repo_root / "results" / "rq_checkpoint_audit.json"))
    if set(audit["missing_cells"]) != MISSING_CONFIGS:
        raise RuntimeError("the task doc's 21 configs and "
                           "results/rq_checkpoint_audit.json disagree")
    order = {d: i for i, d in enumerate(DESIGN_ORDER)}
    cells.sort(key=lambda c: (order[(c["backbone"], c["adapter"])],
                              c["seed"], c["head"] != "evidential"))
    if only:
        cells = [c for c in cells if all(c.get(k) == v for k, v in only.items())]
    return cells


# =====================================================================
# Part A: pre-flight
# =====================================================================
#: Committed files this notebook must never change. `configs/` covers the two
#: frozen episode files and every grid recipe.
FROZEN_PATHS = ("configs", "results/grid", "results/mvt_results.json",
                "results/rq_summary.json", "results/rq_checkpoint_audit.json")


def frozen_files_untouched(repo_root: Path) -> dict:
    r = subprocess.run(["git", "-C", str(repo_root), "status", "--porcelain",
                        "--", *FROZEN_PATHS], capture_output=True, text=True)
    if r.returncode != 0:
        return {"ok": None, "error": r.stderr.strip(), "paths": list(FROZEN_PATHS)}
    changed = [ln for ln in r.stdout.splitlines() if ln.strip()]
    return {"ok": not changed, "changed": changed, "paths": list(FROZEN_PATHS)}


def preflight(repo_root: Path, cells: list[dict], log=print) -> dict:
    """Checks that cost no GPU time. Any `problems` entry means do not train."""
    import yaml

    problems, notes = [], []
    val = list(yaml.safe_load(open(repo_root / "configs/val_episodes.yaml"))["seeds"])
    test = list(yaml.safe_load(open(repo_root / "configs/test_episodes.yaml"))["seeds"])
    if val != list(range(10000, 10100)):
        problems.append(f"VAL seeds are {val[0]}..{val[-1]} (n={len(val)}), "
                        f"expected 10000..10099")
    if test != list(range(0, 600)):
        problems.append(f"TEST seeds are {test[0]}..{test[-1]} (n={len(test)}), "
                        f"expected 0..599")
    if set(val) & set(test):
        problems.append("VAL and TEST seeds overlap")

    smoke = []
    for c in cells:
        if not (repo_root / c["config"]).exists():
            problems.append(f"missing config {c['config']}")
        committed = repo_root / c["results_json"]
        if not committed.exists():
            problems.append(f"missing committed baseline {c['results_json']}")
            continue
        n = int(json.load(open(committed)).get("num_episodes", -1))
        if n != EXPECTED_N_EPISODES:
            smoke.append({"config": c["config"], "committed_num_episodes": n})
    unexpected = {s["config"] for s in smoke} - EXPECTED_SMOKE_BASELINES
    if unexpected:
        notes.append(f"unexpected non-600-episode baselines: {sorted(unexpected)}")

    frozen = frozen_files_untouched(repo_root)
    if frozen["ok"] is False:
        problems.append(f"frozen paths already modified in this clone: {frozen['changed']}")

    log(f"  VAL seeds {val[0]}..{val[-1]} (n={len(val)}), "
        f"TEST seeds {test[0]}..{test[-1]} (n={len(test)}), disjoint={not set(val) & set(test)}")
    log(f"  {len(cells)} cell(s): configs + committed baselines present = "
        f"{not any('missing' in p for p in problems)}")
    for s in smoke:
        log(f"  NOTE baseline is a smoke run: {s['config']} "
            f"(num_episodes={s['committed_num_episodes']}) -> first-"
            f"{s['committed_num_episodes']}-episode guard applies (Section 8)")
    log(f"  frozen paths clean: {frozen['ok']}")
    for p in problems:
        log(f"  PROBLEM: {p}")
    for n_ in notes:
        log(f"  NOTE: {n_}")
    return {"ok": not problems, "problems": problems, "notes": notes,
            "smoke_baselines": smoke, "frozen": frozen}


# =====================================================================
# Part A: restore what an earlier session (or Step 10) already paid for
# =====================================================================
#: Directories never worth walking: huge image trees, VCS metadata.
_SKIP_DIRS = frozenset({".git", "tiny-imagenet-200", "cifar-100-python",
                        "wandb", "__pycache__"})


def restore_session_artifacts(repo_root: Path, cells: list[dict],
                              search_roots=("/kaggle/input",), log=print) -> dict:
    """Scan attached inputs (zips and loose files) and restore, for the given
    cells only:

      - checkpoints/model_phase2_*.pt   (task A.1 recovery, and resume)
      - results/rq_factorial/<cell>.json, accepted only if it is a full
        600-episode record, so a smoke output can never enter the aggregation
      - results/rq_completion/provenance/<cell>.json and logs/*.log

    Targeted on purpose: rq_drivers.recover_checkpoints extracts EVERY
    checkpoint in the Step 10 zips (99 files, ~2.6 GB, staged then copied) and
    none of those 99 are needed here. Their scores are already in the
    committed rq_factorial JSONs. Existing files are never overwritten, and
    results/grid/ is never written.
    """
    ckpt_dir = repo_root / "checkpoints"
    fac_dir = repo_root / "results" / "rq_factorial"
    comp_dir = repo_root / "results" / OUT_SUBDIR
    want_ckpt = {Path(c["checkpoint"]).name for c in cells}
    want_ids = {cell_id(c) for c in cells}
    got = {"checkpoints": [], "factorial": [], "provenance": [], "logs": [],
           "rejected": []}
    zips = []

    def _route(member: str):
        p = Path(member)
        if p.suffix == ".pt" and p.name in want_ckpt:
            return "checkpoints", ckpt_dir
        if p.suffix == ".json" and p.stem in want_ids:
            if p.parent.name == "rq_factorial":
                return "factorial", fac_dir
            if p.parent.name == "provenance" and p.parent.parent.name == OUT_SUBDIR:
                return "provenance", comp_dir / "provenance"
        if (p.suffix == ".log" and p.parent.name == "logs"
                and p.parent.parent.name == OUT_SUBDIR):
            return "logs", comp_dir / "logs"
        return None

    def _take(kind: str, dest_dir: Path, name: str, reader) -> None:
        dest = dest_dir / name
        if dest.exists():
            return
        dest_dir.mkdir(parents=True, exist_ok=True)
        tmp = dest_dir / (name + ".part")
        with open(tmp, "wb") as out:
            shutil.copyfileobj(reader, out)
        if kind == "factorial":
            try:
                rec = json.load(open(tmp))
                ok = int(rec["summary"]["num_episodes"]) == EXPECTED_N_EPISODES
            except Exception:  # noqa: BLE001 -- unreadable means rejected
                ok = False
            if not ok:
                tmp.unlink()
                got["rejected"].append(name)
                return
        tmp.rename(dest)
        got[kind].append(name)

    for root in search_roots:
        rp = Path(root)
        if not rp.exists():
            log(f"  (no such path: {root})")
            continue
        for dirpath, dirnames, filenames in os.walk(rp):
            dirnames[:] = sorted(d for d in dirnames if d not in _SKIP_DIRS)
            for fn in sorted(filenames):
                p = Path(dirpath) / fn
                if p.suffix == ".zip":
                    zips.append(str(p))
                    try:
                        with zipfile.ZipFile(p) as zf:
                            for m in zf.namelist():
                                route = _route(m)
                                if route:
                                    with zf.open(m) as fh:
                                        _take(route[0], route[1], Path(m).name, fh)
                    except zipfile.BadZipFile:
                        log(f"  {p.name}: not a readable zip, skipped")
                    continue
                route = _route(str(p))
                if route:
                    with open(p, "rb") as fh:
                        _take(route[0], route[1], p.name, fh)

    log(f"  scanned {list(search_roots)}: {len(zips)} zip(s)")
    log(f"  restored: {len(got['checkpoints'])} checkpoint(s), "
        f"{len(got['factorial'])} factorial record(s), "
        f"{len(got['provenance'])} provenance file(s), {len(got['logs'])} log(s)")
    if got["rejected"]:
        log(f"  rejected (not a 600-episode record): {got['rejected']}")
    return {"zips_seen": zips, **{k: sorted(v) for k, v in got.items()}}


def _step10_wall_seconds(repo_root: Path) -> dict[str, float]:
    """Measured train+eval wall time per config, from the Step 10 run log."""
    out: dict[str, float] = {}
    p = repo_root / "results" / "grid" / "_run_log.jsonl"
    if p.exists():
        for line in open(p):
            e = json.loads(line)
            if e.get("status") == "ok" and e.get("wall_seconds"):
                out[e["config"]] = float(e["wall_seconds"])
    return out


def coverage(repo_root: Path, cells: list[dict]) -> dict:
    """What this session actually has left to do, with a measured estimate."""
    fac_dir = repo_root / "results" / "rq_factorial"
    wall = _step10_wall_seconds(repo_root)
    out = {"n_cells": len(cells), "evaluated": [], "checkpoint_present": [],
           "needs_training": [], "estimate_minutes": 0.0}
    eval_s = 240.0  # measured: Phase A scored cifar 5-shot cells in ~190-230 s
    for c in cells:
        cid = cell_id(c)
        if (fac_dir / f"{cid}.json").exists():
            out["evaluated"].append(cid)
        elif (repo_root / c["checkpoint"]).exists():
            out["checkpoint_present"].append(cid)
            out["estimate_minutes"] += eval_s / 60
        else:
            out["needs_training"].append(cid)
            # Step 10 wall already includes one evaluation; factorial replaces it.
            out["estimate_minutes"] += wall.get(c["config"], 1500.0) / 60
    out["estimate_minutes"] = round(out["estimate_minutes"], 1)
    return out


def record_provenance(repo_root: Path, cells: list[dict], had_checkpoint: set,
                      session_info: dict, log=print) -> list[str]:
    """After run_phase_a: note, per newly evaluated cell, whether THIS session
    trained its checkpoint or found one on disk. The best_val_epoch check in
    the guard table only tests training reproducibility for trained cells."""
    fac_dir = repo_root / "results" / "rq_factorial"
    prov_dir = repo_root / "results" / OUT_SUBDIR / "provenance"
    prov_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for c in cells:
        cid = cell_id(c)
        if (prov_dir / f"{cid}.json").exists() or not (fac_dir / f"{cid}.json").exists():
            continue
        source = ("checkpoint_found_on_disk" if cid in had_checkpoint
                  else "trained_this_session")
        with open(prov_dir / f"{cid}.json", "w") as f:
            json.dump({"cell": cid, "checkpoint_source": source, **session_info},
                      f, indent=2, sort_keys=True)
        written.append(cid)
    log(f"  provenance written for {len(written)} cell(s)")
    return written


# =====================================================================
# Part A.4: regression guards
# =====================================================================
def smoke_baseline_guards(repo_root: Path, cells: list[dict], *, device,
                          log=print) -> list[dict]:
    """For every cell whose committed baseline is not a 600-episode run,
    re-score the checkpoint on exactly the committed number of test seeds and
    diff against it with the unchanged rq_core.regression_guard."""
    import rq_drivers as D

    out_dir = repo_root / "results" / OUT_SUBDIR / "smoke_baseline_guard"
    results = []
    for c in cells:
        committed_path = repo_root / c["results_json"]
        committed = json.load(open(committed_path))
        n = int(committed.get("num_episodes", EXPECTED_N_EPISODES))
        if n == EXPECTED_N_EPISODES:
            continue
        cid = cell_id(c)
        out_json = out_dir / f"{cid}__first{n}episodes.json"
        ckpt = repo_root / c["checkpoint"]
        if out_json.exists():
            rec = json.load(open(out_json))
        elif not ckpt.exists():
            log(f"  {cid}: no checkpoint yet -- smoke-baseline guard deferred")
            results.append({"cell": cid, "committed_num_episodes": n,
                            "guard": {"status": "no_checkpoint_yet"}})
            continue
        else:
            log(f"  {cid}: re-scoring the checkpoint on test seeds 0..{n - 1} "
                f"against the committed {n}-episode baseline")
            rec = D.factorial_run_one(
                repo_root, repo_root / c["config"], ckpt, out_json,
                device=device, num_episodes=n, logits_out=None,
                committed_metrics=committed_path,
                meta={**c, "purpose": "smoke_baseline_guard"}, log=log)
        g = rec["regression_guard"]
        log(f"    guard: {g.get('status')} ({g.get('n_exact', 0)}/{g.get('n_keys', 0)} "
            f"exact, max|diff|={g.get('max_abs_diff', float('nan')):.2e})")
        results.append({
            "cell": cid, "committed_num_episodes": n, "guard": g,
            "best_val_epoch": {"committed": committed.get("best_val_epoch"),
                               "checkpoint": rec.get("best_val_epoch")},
            "n_params": {"committed": committed.get("n_params"),
                         "checkpoint": rec.get("n_params")},
        })
    return results


def guard_table(repo_root: Path, cells: list[dict], smoke_guards: list[dict]) -> dict:
    """One row per target cell: the APPLICABLE guard (600-episode, or the
    first-n-episode one when the baseline is a smoke run), plus two checks
    that are independent of evaluation: best_val_epoch and n_params."""
    fac_dir = repo_root / "results" / "rq_factorial"
    prov_dir = repo_root / "results" / OUT_SUBDIR / "provenance"
    sg = {s["cell"]: s for s in smoke_guards}
    rows = []
    for c in cells:
        cid = cell_id(c)
        rec_path = fac_dir / f"{cid}.json"
        if not rec_path.exists():
            rows.append({"cell": cid, "guard_status": "not_run"})
            continue
        rec = json.load(open(rec_path))
        committed = json.load(open(repo_root / c["results_json"]))
        prov = (json.load(open(prov_dir / f"{cid}.json"))
                if (prov_dir / f"{cid}.json").exists() else {})
        n_comm = int(committed.get("num_episodes", -1))
        if n_comm == EXPECTED_N_EPISODES:
            g, baseline = rec["regression_guard"], "600-episode"
        else:
            g = sg.get(cid, {}).get("guard", {"status": "smoke_guard_missing"})
            baseline = f"first-{n_comm}-episode (committed JSON is a smoke run)"
        bve_c, bve_r = committed.get("best_val_epoch"), rec.get("best_val_epoch")
        rows.append({
            "cell": cid,
            "checkpoint_source": prov.get("checkpoint_source", "unknown"),
            "gpu": prov.get("gpu"),
            "baseline": baseline,
            "guard_status": g.get("status"),
            "n_exact": g.get("n_exact"), "n_keys": g.get("n_keys"),
            "max_abs_diff": g.get("max_abs_diff"),
            "max_abs_diff_key": g.get("max_abs_diff_key"),
            "guard_600_episode_status": rec["regression_guard"].get("status"),
            "best_val_epoch_committed": bve_c, "best_val_epoch_new": bve_r,
            "best_val_epoch_match": (bve_c == bve_r) if bve_c is not None else None,
            "n_params_match": committed.get("n_params") == rec.get("n_params"),
        })
    counts: dict[str, int] = {}
    for r in rows:
        counts[r["guard_status"]] = counts.get(r["guard_status"], 0) + 1
    evaluated = [r for r in rows if r["guard_status"] != "not_run"]
    trained = [r for r in evaluated if r.get("checkpoint_source") == "trained_this_session"]
    return {
        "rows": rows,
        "status_counts": counts,
        "n_evaluated": len(evaluated),
        "n_reproduced": sum(1 for r in evaluated
                            if r["guard_status"] in ("exact", "within_tol")),
        "n_best_val_epoch_match": sum(1 for r in evaluated if r.get("best_val_epoch_match")),
        "n_trained_this_session": len(trained),
        "n_params_all_match": (all(r.get("n_params_match") for r in evaluated)
                               if evaluated else None),
    }


def guards_markdown(g: dict) -> str:
    L = ["### Regression guard against the committed Step 10 grid", "",
         f"Evaluated {g['n_evaluated']}/21. Guard status counts: "
         + ", ".join(f"`{k}` {v}" for k, v in sorted(g["status_counts"].items()))
         + f". Reproduced (exact or within 1e-6): **{g['n_reproduced']}/{g['n_evaluated']}**. "
         f"`best_val_epoch` matches: **{g['n_best_val_epoch_match']}/{g['n_evaluated']}**. "
         f"`n_params` all match: {g['n_params_all_match']}.", "",
         "| cell | checkpoint | baseline | guard | exact/keys | max abs diff | best_val_epoch committed → new |",
         "|---|---|---|---|---:|---:|---:|"]
    for r in g["rows"]:
        if r["guard_status"] == "not_run":
            L.append(f"| `{r['cell']}` | | | not run | | | |")
            continue
        mad = r.get("max_abs_diff")
        L.append(f"| `{r['cell']}` | {r['checkpoint_source']} | {r['baseline']} | "
                 f"`{r['guard_status']}` | {r.get('n_exact')}/{r.get('n_keys')} | "
                 f"{'' if mad is None else f'{mad:.2e}'} | "
                 f"{r['best_val_epoch_committed']} → {r['best_val_epoch_new']} |")
    return "\n".join(L)


# =====================================================================
# Part A.5: RQ2 (code: rq1_verdict) and RQ4 (code: rq2_verdict)
# =====================================================================
def rq2_completeness(recs: list[dict], grid_cells: list[dict]) -> dict:
    """Is the objective x score design FULLY crossed over the designs present?

    eta_squared's `balanced` only sees cells that have observations. This
    enumerates what should be there, so a wholly missing arm is reported
    rather than silently leaving the design out.
    """
    seeds = sorted({c["seed"] for c in grid_cells})
    designs = sorted({design_of(c) for c in grid_cells})
    present: dict[tuple, set] = {}
    for r in recs:
        present.setdefault((A.design_key(r), r["interpretation"]), set()).add(r["seed"])

    complete, absent, half, partial = [], [], [], []
    for d in designs:
        ev = present.get((d, "evidential"), set())
        sm = present.get((d, "softmax"), set())
        if ev == set(seeds) and sm == set(seeds):
            complete.append(d)
        elif not ev and not sm:
            absent.append(d)
        elif not ev or not sm:
            half.append({"design": d, "evidential_seeds": sorted(ev),
                         "softmax_seeds": sorted(sm)})
        else:
            partial.append({"design": d, "evidential_seeds": sorted(ev),
                            "softmax_seeds": sorted(sm)})

    rows = A.factorial_observations(recs)
    zero_cells, count_values = {}, {}
    for g in ("far", "near"):
        cnt: dict[tuple, int] = {}
        for r in rows:
            if r["pool_group"] == g:
                k = (r["design"], r["objective"], r["score"])
                cnt[k] = cnt.get(k, 0) + 1
        seen = sorted({k[0] for k in cnt})
        zero_cells[g] = [f"{d} | {o} | {s}" for d in seen
                         for o in ("evidential", "softmax") for s in A.RQ1_SCORES
                         if (d, o, s) not in cnt]
        count_values[g] = sorted(set(cnt.values()))
    return {
        "n_designs_expected": len(designs), "complete_designs": complete,
        "absent_designs": absent, "half_present_designs": half,
        "partial_seed_designs": partial, "zero_observation_cells": zero_cells,
        "observation_counts_per_cell": count_values,
        "fully_crossed": (not half and not partial
                          and not any(zero_cells.values())
                          and all(len(v) == 1 for v in count_values.values())),
    }


def _max_numeric_diff(a, b, path="") -> tuple[float, list[str]]:
    if isinstance(a, dict) and isinstance(b, dict):
        worst, bad = 0.0, []
        for k in sorted(set(a) | set(b)):
            if k not in a or k not in b:
                bad.append(f"{path}/{k}: present on one side only")
                continue
            w, bb = _max_numeric_diff(a[k], b[k], f"{path}/{k}")
            worst, bad = max(worst, w), bad + bb
        return worst, bad
    if isinstance(a, bool) or isinstance(b, bool) or not (
            isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return 0.0, ([] if a == b else [f"{path}: {a!r} != {b!r}"])
    if a == b:
        return 0.0, []
    if math.isnan(a) and math.isnan(b):
        return 0.0, []
    return abs(float(a) - float(b)), []


def rq2_rq4_aggregate(repo_root: Path, grid_cells: list[dict], new_ids: set,
                      committed_summary_path: Path) -> dict:
    """Task A.5, steps 1-3 and 5, over whatever records are on disk."""
    known = {cell_id(c) for c in grid_cells}
    recs = [r for r in A.load_records(repo_root / "results" / "rq_factorial")
            if cell_id(r["meta"]) in known]
    old = [r for r in recs if cell_id(r["meta"]) not in new_ids]
    new = [r for r in recs if cell_id(r["meta"]) in new_ids]
    committed = json.load(open(committed_summary_path))

    # 0. Before trusting any new number: the unchanged aggregator on the
    #    records that produced the committed summary must reproduce it.
    old_v2 = A.rq1_verdict(A.factorial_observations(old))
    old_v4 = A.rq2_verdict(A.rq2_table(old))
    d2, bad2 = _max_numeric_diff(old_v2, committed["rq1_verdict"])
    d4, bad4 = _max_numeric_diff(old_v4, committed["rq2_verdict"])

    comp = rq2_completeness(recs, grid_cells)
    rows = A.factorial_observations(recs)
    after = A.rq1_verdict(rows)
    keep = set(comp["complete_designs"])
    after_complete = A.rq1_verdict([r for r in rows if r["design"] in keep])

    rq4_rows = A.rq2_table(recs)
    return {
        "coverage": {
            "n_records": len(recs), "n_grid_cells": len(grid_cells),
            "n_committed_records": len(old), "n_new_records": len(new),
            "new_records": sorted(cell_id(r["meta"]) for r in new),
            "n_evidential_records": sum(1 for r in recs if r["interpretation"] == "evidential"),
        },
        "aggregator_reproduces_committed": {
            "rq2_max_abs_diff": d2, "rq2_non_numeric_mismatches": bad2,
            "rq4_max_abs_diff": d4, "rq4_non_numeric_mismatches": bad4,
            "ok": d2 <= 1e-12 and d4 <= 1e-12 and not bad2 and not bad4,
        },
        "completeness": comp,
        "rq2_before_committed": committed["rq1_verdict"],
        "rq2_after_all_records": after,
        "rq2_after_complete_designs_only": after_complete,
        "rq2_quotable": ("all_records" if comp["fully_crossed"]
                         else "complete_designs_only"),
        "rq2_tables_after": {g: A.table_2x4(rows, g) for g in ("far", "near")},
        "rq4_before_committed": committed["rq2_verdict"],
        "rq4_after": A.rq2_verdict(rq4_rows),
        "rq4_new_cells_only": A.rq2_verdict(A.rq2_table(new)) if new else None,
        "rq4_rows": rq4_rows,
    }


def rq2_rq4_markdown(agg: dict) -> str:
    cov, comp, rep = agg["coverage"], agg["completeness"], agg["aggregator_reproduces_committed"]
    L = ["## RQ2 / RQ4 re-aggregation", "",
         f"Phase A factorial records: **{cov['n_records']}/{cov['n_grid_cells']}** "
         f"({cov['n_committed_records']} committed + {cov['n_new_records']} new). "
         f"Evidential records: {cov['n_evidential_records']}.", "",
         f"Unchanged aggregator reproduces the committed `results/rq_summary.json` on the "
         f"committed records: **{rep['ok']}** (RQ2 max abs diff {rep['rq2_max_abs_diff']:.1e}, "
         f"RQ4 {rep['rq4_max_abs_diff']:.1e}).", "",
         f"Design completeness: {len(comp['complete_designs'])}/{comp['n_designs_expected']} "
         f"complete, {len(comp['absent_designs'])} absent, "
         f"{len(comp['half_present_designs'])} with one objective arm missing, "
         f"{len(comp['partial_seed_designs'])} with seeds missing. Zero-observation "
         f"objective×score cells: far {len(comp['zero_observation_cells']['far'])}, "
         f"near {len(comp['zero_observation_cells']['near'])}. "
         f"**Fully crossed: {comp['fully_crossed']}.**"]
    for h in comp["half_present_designs"]:
        L.append(f"- one arm missing: `{h['design']}` evidential seeds "
                 f"{h['evidential_seeds']}, softmax seeds {h['softmax_seeds']}")
    L += ["", "### RQ2: objective vs scoring rule (η² share of OOD-AUROC variance)", "",
          "| variant | pool | score η² | objective η² | score/objective | balanced | n |",
          "|---|---|---:|---:|---:|---|---:|"]
    variants = [("before: committed, 99 records", agg["rq2_before_committed"]),
                ("after: all records", agg["rq2_after_all_records"])]
    if not comp["fully_crossed"]:
        variants.append(("after: complete designs only", agg["rq2_after_complete_designs_only"]))
    for label, v in variants:
        for g in ("far", "near"):
            if g not in v:
                continue
            e = v[g]["eta_squared"]
            L.append(f"| {label} | {g} | {100 * e['score']:.2f}% | "
                     f"{100 * e['objective']:.3f}% | "
                     f"{v[g]['ratio_score_over_objective']:.1f}× | "
                     f"{v[g]['balanced']} | {v[g]['n']} |")
    L += ["", f"Quotable variant: **{agg['rq2_quotable'].replace('_', ' ')}**"
          + ("" if comp["fully_crossed"] else
             " (the all-records row still contains a half-present design; "
             "`balanced` does not detect that)."), ""]
    for g in ("far", "near"):
        t = agg["rq2_tables_after"][g]
        L += [f"Mean AUROC, {g}-OOD, after (rows = training objective):", "",
              "| trained as | " + " | ".join(A.RQ1_SCORES) + " |",
              "|---|" + "---:|" * len(A.RQ1_SCORES)]
        for obj in ("evidential", "softmax"):
            L.append(f"| {obj} | " + " | ".join(
                f"{t[obj][s]['mean']:.4f} (n={t[obj][s]['n']})" for s in A.RQ1_SCORES) + " |")
        L.append("")
    L += ["### RQ4: post-hoc evidence-affine refit (VAL only)", "",
          "| variant | evidential cells | ECE improved | AUROC preserved (Δ ≥ −0.005) | "
          "mean ΔECE | mean ΔAUROC | min Spearman ρ | reordering observed |",
          "|---|---:|---:|---:|---:|---:|---:|---|"]
    for label, v in (("before: committed", agg["rq4_before_committed"]),
                     ("after: all records", agg["rq4_after"]),
                     ("new cells only", agg["rq4_new_cells_only"])):
        if not v or "error" in v:
            continue
        L.append(f"| {label} | {v['n_cells']} | {v['ece_improved_in']}/{v['n_cells']} | "
                 f"{v['auroc_preserved_in']}/{v['auroc_comparisons']} | "
                 f"{v['ece_mean_delta']:+.4f} | {v['auroc_mean_delta']:+.4f} | "
                 f"{v['min_spearman_rho']:.4f} | {v['reordering_ever_observed']} |")
    return "\n".join(L)


# =====================================================================
# Part B: RQ1 two-way interactions
# =====================================================================
FACTORS = ("dataset", "k_shot", "backbone", "adapter", "head")
BASELINE_ADAPTERS = frozenset({"full_ft", "linear_probe"})

#: Outcome -> results/mvt_results.json key. Checked 2026-09-14 by reproducing
#: every published main-effect share in docs/RQ_RESULTS_SUMMARY.md §3.1:
#:  - ECE is `ece_pooled`, NOT `ece_per_episode_mean` as the task doc guessed.
#:    That key gives head 83.03 / residual 10.55, not the published 82.89 / 7.70.
#:  - "OOD AUROC (far)" is the SVHN pool alone under each head's native score
#:    (== `ood_auroc_mean`, the legacy primary pool), not an SVHN+Gaussian mean.
#:  - native score = vacuity (evidential) / msp (softmax): Step 10 scored each
#:    head with its own score only.
RQ1_OUTCOMES = {
    "accuracy": "accuracy_mean",
    "ece": "ece_pooled",
    "far_ood_svhn": "ood_auroc__svhn_far__{native}",
    "near_ood_tin": "ood_auroc__tin_near__{native}",
}

#: docs/RQ_RESULTS_SUMMARY.md §3.1, in percent. The regression target for
#: `flatten`: if these do not come back, the row table is wrong.
PUBLISHED_MAIN_EFFECTS = {
    "accuracy": {"dataset": 1.75, "k_shot": 76.05, "backbone": 3.92,
                 "adapter": 9.14, "head": 0.19, "residual": 8.95},
    "ece": {"dataset": 2.02, "k_shot": 2.13, "backbone": 4.63,
            "adapter": 0.63, "head": 82.89, "residual": 7.70},
    "far_ood_svhn": {"dataset": 1.18, "k_shot": 22.68, "backbone": 11.54,
                     "adapter": 1.73, "head": 39.01, "residual": 23.87},
    "near_ood_tin": {"dataset": 1.64, "k_shot": 42.60, "backbone": 0.61,
                     "adapter": 21.98, "head": 20.98, "residual": 11.42},
}

#: eta_squared names a pair in FACTORS order, so the task doc's
#: "adapter:backbone" is this key.
FOCUS_TERM = "backbone:adapter"

#: For an evidential factorial record, the keys holding the same quantity as
#: each outcome. These are exactly the pairs rq_core.regression_guard maps.
FACTORIAL_KEYS_EVIDENTIAL = {
    "accuracy": "accuracy_mean__evidential_native",
    "ece": "ece_pooled__evidential_native",
    "far_ood_svhn": "ood_auroc__svhn_far__vacuity_native",
    "near_ood_tin": "ood_auroc__tin_near__vacuity_native",
}
SMOKE_CELL = {"dataset": "cifar_fs", "k_shot": 5, "backbone": "mobilenetv3_small",
              "adapter": "lora", "head": "evidential", "seed": 42}


def _obs_key(dataset, k_shot, backbone, adapter, head, seed) -> tuple:
    return (dataset, int(k_shot), backbone, adapter, head, int(seed))


def flatten(mvt: dict, outcome: str, overrides: dict | None = None) -> list[dict]:
    """One row per (cell, seed) of the balanced 2^5 design (baselines excluded).

    `overrides` maps an `_obs_key` to {outcome: value}. It replaces or supplies
    that single observation.
    """
    tmpl = RQ1_OUTCOMES[outcome]
    rows = []
    for dataset, by_shot in mvt["results"].items():
        for shot_key, by_backbone in by_shot.items():
            k_shot = int(shot_key.replace("shot", ""))
            for backbone, by_adapter in by_backbone.items():
                for adapter, by_head in by_adapter.items():
                    if adapter in BASELINE_ADAPTERS:
                        continue
                    for head, leaf in by_head.items():
                        key = tmpl.format(native="vacuity" if head == "evidential" else "msp")
                        per_seed = {int(s): float(v)
                                    for s, v in (leaf.get(key) or {}).get("per_seed", {}).items()}
                        for ok, vals in (overrides or {}).items():
                            if ok[:5] == (dataset, k_shot, backbone, adapter, head) and outcome in vals:
                                per_seed[ok[5]] = float(vals[outcome])
                        for seed in sorted(per_seed):
                            rows.append({"dataset": dataset, "k_shot": k_shot,
                                         "backbone": backbone, "adapter": adapter,
                                         "head": head, "seed": seed,
                                         "value": per_seed[seed]})
    return rows


def _coded_columns(rows: list[dict], interactions: bool) -> tuple[dict, list[str]]:
    cols = {}
    for f in FACTORS:
        levels = sorted({r[f] for r in rows}, key=str)
        if len(levels) != 2:
            raise ValueError(f"factor {f!r} has {len(levels)} levels; this "
                             f"decomposition assumes the 2^5 design")
        cols[f] = np.array([1.0 if r[f] == levels[1] else -1.0 for r in rows])
    terms = list(FACTORS)
    if interactions:
        for a, b in combinations(FACTORS, 2):
            cols[f"{a}:{b}"] = cols[a] * cols[b]
            terms.append(f"{a}:{b}")
    return cols, terms


def ols_type2(rows: list[dict], *, interactions: bool, Y: np.ndarray | None = None) -> dict:
    """Type-II sums of squares, as shares of SS_total, for every row of Y.

    SS(T) = RSS(terms not containing T) - RSS(those terms + T). With +-1 coding
    on a balanced 2-level design every column is orthogonal and this equals
    the classical decomposition eta_squared computes, to machine precision.
    Returns (B,)-shaped arrays when Y is (B, n); a single y otherwise.
    """
    y = np.array([r["value"] for r in rows], dtype=float)
    n = len(y)
    Ys = y[None, :] if Y is None else np.asarray(Y, dtype=float)
    cols, terms = _coded_columns(rows, interactions)
    makers: dict[tuple, np.ndarray] = {}

    def rss(ts):
        key = tuple(sorted(ts))
        if key not in makers:
            X = np.column_stack([np.ones(n)] + [cols[t] for t in key])
            makers[key] = np.eye(n) - X @ np.linalg.pinv(X)
        return np.einsum("bi,ij,bj->b", Ys, makers[key], Ys)

    sst = ((Ys - Ys.mean(axis=1, keepdims=True)) ** 2).sum(axis=1)
    rss_full = rss(terms)
    ss = {}
    for t in terms:
        parts = set(t.split(":"))
        reduced = [u for u in terms if not parts <= set(u.split(":"))]
        ss[t] = rss(reduced) - rss(reduced + [t])
    X_full = np.column_stack([np.ones(n)] + [cols[t] for t in terms])
    df_resid = n - int(np.linalg.matrix_rank(X_full))
    shares = {t: ss[t] / sst for t in terms}
    shares["residual"] = rss_full / sst
    return {"terms": terms, "shares": shares, "ss": ss, "rss_full": rss_full,
            "sst": sst, "n": n, "df_resid": df_resid}


def seed_bootstrap(rows: list[dict], n_boot: int, seed: int) -> np.ndarray:
    """(n_boot, n) resampled outcomes: the seeds of each of the 32 cells are
    redrawn with replacement, the design itself is held fixed (task B.3)."""
    rng = np.random.default_rng(seed)
    y = np.array([r["value"] for r in rows], dtype=float)
    groups: dict[tuple, list[int]] = {}
    for i, r in enumerate(rows):
        groups.setdefault(tuple(r[f] for f in FACTORS), []).append(i)
    idx = np.empty((n_boot, len(rows)), dtype=int)
    for members in groups.values():
        g = np.asarray(members)
        idx[:, g] = g[rng.integers(0, len(g), size=(n_boot, len(g)))]
    return y[idx]


def _f_test_p(ss_term: float, rss_full: float, df_resid: int) -> float | None:
    try:
        from scipy import stats
    except ImportError:
        return None
    if df_resid <= 0 or rss_full <= 0:
        return None
    return float(stats.f.sf(ss_term / (rss_full / df_resid), 1, df_resid))


def decompose_outcome(rows: list[dict], *, n_boot: int, boot_seed: int) -> dict:
    eta_main = A.eta_squared(rows, list(FACTORS), value="value", interactions=False)
    eta_int = A.eta_squared(rows, list(FACTORS), value="value", interactions=True)
    ols_main = ols_type2(rows, interactions=False)
    ols_int = ols_type2(rows, interactions=True)
    balanced = bool(eta_int["balanced"])
    ols_int_pt = {k: float(v[0]) for k, v in ols_int["shares"].items()}
    ols_main_pt = {k: float(v[0]) for k, v in ols_main["shares"].items()}

    agreement = max(abs(eta_int["eta_squared"][k] - ols_int_pt[k]) for k in ols_int_pt)
    if balanced and agreement > 1e-9:
        raise RuntimeError(f"OLS Type II and eta_squared disagree by {agreement:.2e} "
                           f"on a BALANCED outcome -- the OLS path is wrong, stop")
    if balanced:
        point, main_only = dict(eta_int["eta_squared"]), dict(eta_main["eta_squared"])
        method = "eta_squared (exact: balanced design)"
    else:
        point, main_only = ols_int_pt, ols_main_pt
        method = "OLS Type II (eta_squared is not exact on unbalanced rows)"

    boot = (ols_type2(rows, interactions=True, Y=seed_bootstrap(rows, n_boot, boot_seed))
            if n_boot else None)
    interaction_terms = [f"{a}:{b}" for a, b in combinations(FACTORS, 2)]
    table = []
    for t in list(FACTORS) + interaction_terms + ["residual"]:
        row = {"term": t, "kind": ("residual" if t == "residual" else
                                   "interaction" if ":" in t else "main"),
               "share": float(point[t])}
        if boot is not None:
            b = boot["shares"][t]
            row.update(boot_lo=float(np.percentile(b, 2.5)),
                       boot_hi=float(np.percentile(b, 97.5)),
                       boot_sd=float(np.std(b)))
        if t != "residual":
            row["f_test_p"] = _f_test_p(float(ols_int["ss"][t][0]),
                                        float(ols_int["rss_full"][0]),
                                        ols_int["df_resid"])
        table.append(row)

    resid_main = float(main_only["residual"])
    resid_int = float(point["residual"])
    focus = float(point[FOCUS_TERM])
    return {
        "n": len(rows), "balanced": balanced, "method": method,
        "df_resid_with_interactions": ols_int["df_resid"],
        "ols_vs_eta_squared_max_abs_diff": float(agreement),
        "main_effects_only": {k: float(v) for k, v in main_only.items()},
        "with_two_way": {k: float(v) for k, v in point.items()},
        "eta_squared_with_two_way_raw": {k: float(v) for k, v in eta_int["eta_squared"].items()},
        "table": table,
        "residual_main_effects_only": resid_main,
        "residual_with_two_way": resid_int,
        "two_way_total": float(sum(point[t] for t in interaction_terms)),
        "focus_term": FOCUS_TERM, "focus_share": focus,
        "focus_share_of_main_effects_residual": (focus / resid_main if resid_main else None),
        "sum_of_shares_incl_residual": float(sum(point.values())),
    }


def check_published(outcome: str, main_only: dict) -> dict:
    pub = PUBLISHED_MAIN_EFFECTS[outcome]
    diffs = {k: abs(100 * main_only[k] - v) for k, v in pub.items()}
    # published values are rounded to 2 d.p.
    return {"ok": all(d <= 0.0051 for d in diffs.values()),
            "max_abs_diff_pp": max(diffs.values()), "diffs_pp": diffs}


def rq1_interactions(mvt_path: Path, *, overrides: dict | None = None,
                     n_boot: int = 2000, boot_seed: int = 20260914) -> dict:
    mvt = json.load(open(mvt_path))
    out = {"source": str(mvt_path), "factors": list(FACTORS),
           "baseline_adapters_excluded": sorted(BASELINE_ADAPTERS),
           "outcome_keys": RQ1_OUTCOMES, "focus_term": FOCUS_TERM,
           "n_boot": n_boot, "boot_seed": boot_seed,
           "overrides": ({"|".join(map(str, k)): v for k, v in overrides.items()}
                         if overrides else None),
           "outcomes": {}}
    for outcome in RQ1_OUTCOMES:
        rows = flatten(mvt, outcome, overrides)
        dec = decompose_outcome(rows, n_boot=n_boot, boot_seed=boot_seed)
        dec["published_main_effects_check"] = check_published(outcome, dec["main_effects_only"])
        out["outcomes"][outcome] = dec
    return out


def smoke_cell_overrides(repo_root: Path, guards: dict) -> dict | None:
    """The one 20-episode observation in the RQ1 table, replaced by the SAME
    model's 600-episode numbers. Offered only when the first-20-episode guard
    has shown the checkpoint on disk is that model."""
    cid = cell_id(SMOKE_CELL)
    rec_path = repo_root / "results" / "rq_factorial" / f"{cid}.json"
    row = next((r for r in guards["rows"] if r["cell"] == cid), None)
    if (not rec_path.exists() or row is None
            or row.get("guard_status") not in ("exact", "within_tol")):
        return None
    s = json.load(open(rec_path))["summary"]
    if int(s["num_episodes"]) != EXPECTED_N_EPISODES:
        return None
    key = _obs_key(*(SMOKE_CELL[f] for f in FACTORS), SMOKE_CELL["seed"])
    return {key: {o: float(s[k]) for o, k in FACTORIAL_KEYS_EVIDENTIAL.items()}}


def _pct(x):
    return "" if x is None else f"{100 * x:.2f}%"


def _p(p):
    if p is None:
        return ""
    return "<1e-6" if p < 1e-6 else f"{p:.2g}"


def rq1_markdown(res: dict, title: str) -> str:
    L = [f"## {title}", "",
         f"Factors {', '.join(res['factors'])}; baselines excluded "
         f"({', '.join(res['baseline_adapters_excluded'])}). Seed bootstrap: "
         f"{res['n_boot']} resamples, rng seed {res['boot_seed']}."]
    if res["overrides"]:
        L.append(f"Observations replaced: {list(res['overrides'])}")
    for outcome, d in res["outcomes"].items():
        chk = d["published_main_effects_check"]
        pub_line = (f"Published main effects reproduced: **{chk['ok']}** "
                    f"(max abs diff {chk['max_abs_diff_pp']:.3f} pp)."
                    if not res["overrides"] else
                    f"Main effects vs the published table (expected to move): "
                    f"max abs diff {chk['max_abs_diff_pp']:.2f} pp.")
        L += ["", f"### {outcome}: `{res['outcome_keys'][outcome]}`, n={d['n']}, "
                  f"balanced={d['balanced']}", "",
              f"Method: {d['method']}. {pub_line} Residual df with two-way terms: "
              f"{d['df_resid_with_interactions']}.", "",
              f"Residual **{_pct(d['residual_main_effects_only'])}** with main effects only → "
              f"**{_pct(d['residual_with_two_way'])}** after adding the ten two-way terms "
              f"(which together take {_pct(d['two_way_total'])}).", "",
              f"`{d['focus_term']}` = **{_pct(d['focus_share'])}**, which is "
              f"{_pct(d['focus_share_of_main_effects_residual'])} of the main-effects residual.",
              "", "| term | share | 95% seed bootstrap | bootstrap SD | F-test p |",
              "|---|---:|---:|---:|---:|"]
        rows = sorted([r for r in d["table"] if r["kind"] != "residual"],
                      key=lambda r: -r["share"])
        rows += [r for r in d["table"] if r["kind"] == "residual"]
        for r in rows:
            name = f"**{r['term']}**" if r["term"] == d["focus_term"] else r["term"]
            ci = (f"[{_pct(r['boot_lo'])}, {_pct(r['boot_hi'])}]" if "boot_lo" in r else "")
            L.append(f"| {name} | {_pct(r['share'])} | {ci} | "
                     f"{_pct(r.get('boot_sd'))} | {_p(r.get('f_test_p'))} |")
        if not d["balanced"]:
            L.append(f"\nType-II shares on unbalanced rows do not sum to 100% "
                     f"(here {_pct(d['sum_of_shares_incl_residual'])}).")
    return "\n".join(L)


def rq1_focus_markdown(before: dict, after: dict | None) -> str:
    """Compact side-by-side of the headline interaction numbers."""
    L = ["| outcome | n | residual (main only) | residual (+2-way) | all 2-way | "
         f"`{FOCUS_TERM}` [95% CI] | largest 2-way term |",
         "|---|---:|---:|---:|---:|---:|---|"]
    for label, res in (("as committed", before), ("smoke obs. replaced", after)):
        if res is None:
            continue
        for outcome, d in res["outcomes"].items():
            f = next(r for r in d["table"] if r["term"] == FOCUS_TERM)
            top = max((r for r in d["table"] if r["kind"] == "interaction"),
                      key=lambda r: r["share"])
            ci = f" [{_pct(f['boot_lo'])}, {_pct(f['boot_hi'])}]" if "boot_lo" in f else ""
            L.append(f"| {outcome} ({label}) | {d['n']} | {_pct(d['residual_main_effects_only'])} | "
                     f"{_pct(d['residual_with_two_way'])} | {_pct(d['two_way_total'])} | "
                     f"{_pct(d['focus_share'])}{ci} | {top['term']} {_pct(top['share'])} |")
    return "\n".join(L)


# =====================================================================
# Report
# =====================================================================
def write_report(path: Path, *, session: dict, frozen: dict, guards: dict | None,
                 agg: dict | None, rq1: dict | None, rq1_corrected: dict | None) -> str:
    L = ["# RQ2/RQ4 completion + RQ1 interactions: session report", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M:%S')}. GPU `{session.get('gpu')}`, "
         f"torch `{session.get('torch')}`, CUDA `{session.get('cuda')}`, "
         f"repo HEAD `{session.get('repo_head')}`.", "",
         "Thesis numbering throughout: code `rq1_verdict` = **RQ2**, code "
         "`rq2_verdict` = **RQ4**.", "",
         f"Frozen paths untouched ({', '.join(frozen.get('paths', []))}): "
         f"**{frozen.get('ok')}**" + (f" (changed: {frozen['changed']})" if frozen.get("changed") else ""),
         ""]
    if agg:
        cov = agg["coverage"]
        L += ["## The four deliverables", "",
              f"1. **Coverage:** {cov['n_records']}/{cov['n_grid_cells']} Phase A records "
              f"({cov['n_new_records']}/21 new)."]
        q = (agg["rq2_after_all_records"] if agg["rq2_quotable"] == "all_records"
             else agg["rq2_after_complete_designs_only"])
        parts = []
        for g in ("far", "near"):
            if g in q:
                e = q[g]["eta_squared"]
                parts.append(f"{g}: score {100 * e['score']:.2f}% / objective "
                             f"{100 * e['objective']:.3f}% = {q[g]['ratio_score_over_objective']:.1f}×")
        L.append(f"2. **RQ2 ({agg['rq2_quotable'].replace('_', ' ')}):** " + "; ".join(parts)
                 + f". Fully crossed: {agg['completeness']['fully_crossed']}.")
        v4 = agg["rq4_after"]
        if v4 and "error" not in v4:
            L.append(f"3. **RQ4:** ECE improved in {v4['ece_improved_in']}/{v4['n_cells']} evidential "
                     f"cells; AUROC preserved in {v4['auroc_preserved_in']}/{v4['auroc_comparisons']} "
                     f"comparisons (committed: {agg['rq4_before_committed']['ece_improved_in']}/"
                     f"{agg['rq4_before_committed']['n_cells']} and "
                     f"{agg['rq4_before_committed']['auroc_preserved_in']}/"
                     f"{agg['rq4_before_committed']['auroc_comparisons']}).")
    if rq1:
        L.append("4. **RQ1 interactions** (four full tables in the RQ1 section below). "
                 "Headline numbers:")
        L += ["", rq1_focus_markdown(rq1, rq1_corrected), ""]
    if guards:
        L += ["", guards_markdown(guards), ""]
        mism = [r for r in guards["rows"] if r["guard_status"] == "MISMATCH"]
        if mism:
            L += ["> **MISMATCH present.** These retrained cells do not reproduce the committed "
                  "grid numbers. Their Phase A records are still internally valid (each model is "
                  "scored under all four scores), but they are re-trainings, not the Step 10 "
                  "models, and `results/mvt_results.json` still holds the originals. Report "
                  "this before quoting byte-identical reproducibility for this slice.", ""]
    if agg:
        L += [rq2_rq4_markdown(agg), ""]
    if rq1:
        L += [rq1_markdown(rq1, "RQ1: two-way interactions (committed grid, as published)"), ""]
    if rq1_corrected:
        L += [rq1_markdown(rq1_corrected, "RQ1: sensitivity (the 20-episode smoke observation "
                                          "replaced by the same model's 600-episode scores)"), ""]
    text = "\n".join(L)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    return text


def dump_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str)

Writing scripts/rq_completion.py


In [5]:
import importlib, json
import rq_completion
importlib.reload(rq_completion)
import rq_completion as M
from IPython.display import Markdown, display

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT        = REPO / 'results' / M.OUT_SUBDIR
FAC_DIR    = REPO / 'results' / 'rq_factorial'
LOGITS_DIR = REPO / 'results' / 'rq_logits'
(OUT / 'logs').mkdir(parents=True, exist_ok=True)

SESSION_INFO = {
    'session_start': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    'repo_head': REPO_HEAD,
}

# Tee everything to disk. The Step 11 post-mortem (step_writeups/step11.txt §8): a hosted-notebook
# crash left no trace in the saved .ipynb and was only diagnosable because a log survived by luck.
SESSION_LOG = OUT / 'logs' / f"session_{time.strftime('%Y%m%d-%H%M%S')}.log"

def log(*args):
    line = ' '.join(str(a) for a in args)
    print(line, flush=True)
    with open(SESSION_LOG, 'a') as f:
        f.write(line + '\n')

def show(markdown_text):
    with open(SESSION_LOG, 'a') as f:
        f.write(markdown_text + '\n')
    display(Markdown(markdown_text))

GRID    = M.grid_index(REPO)
CELLS   = M.select_cells(REPO)          # all 21, in run order
SEL     = M.select_cells(REPO, ONLY)    # this session's selection
NEW_IDS = {M.cell_id(c) for c in CELLS}
GUARDS = AGG = RQ1 = RQ1_CORR = None

log(f'=== rq-completion session start {SESSION_INFO["session_start"]} ===')
log(json.dumps(SESSION_INFO))
log(f'{len(CELLS)} target cells; {len(SEL)} selected this session')

=== rq-completion session start 2026-09-14T14:54:34 ===
{"session_start": "2026-09-14T14:54:34", "gpu": "Tesla T4", "torch": "2.10.0+cu128", "cuda": "12.8", "cudnn": 91002, "repo_head": "8d706ea"}
21 target cells; 21 selected this session


## 4. Part B: RQ1 two-way interactions (task B.1–B.3; no GPU, no checkpoints)

The existing `rq_aggregate.eta_squared(..., interactions=True)` runs on the per-seed table from
`results/mvt_results.json`: 32 PEFT cells × 3 seeds, with the Full-FT / Linear-Probe baselines
excluded because they break the 2⁵ balance. It returns 5 main effects, 10 two-way terms, and the
residual.

**Checks, in order:**
1. Row counts must be **96 / 96 / 96 / 95**. The 95 is the smoke observation with no TinyImageNet
   key (fact 3).
2. The **main-effects-only** shares must reproduce the published table
   (`docs/RQ_RESULTS_SUMMARY.md` §3.1) to 2 d.p. If they don't, the flattening is wrong and
   nothing below it means anything.
3. On the three balanced outcomes, the OLS Type-II path must match `eta_squared` to 1e-9. The
   module raises otherwise. Only then is Type II used for the unbalanced TinyImageNet outcome.

**Reading the uncertainty (task B.3).** 96 observations minus 16 parameters leaves 80 residual
df, and the ten two-way terms are small next to the main effects. Every term gets:
- **95% seed-bootstrap interval:** each cell's 3 seeds are redrawn with replacement 2000 times
  and everything is recomputed. With only 3 seeds, a bootstrap understates within-cell variance
  (by a factor of (n−1)/n = 2/3). The residual's interval therefore sits low and the structured
  terms' intervals slightly high. Read the intervals as a *lower bound* on the uncertainty.
- **F-test p:** each 1-df term against the residual left after all main and two-way terms.
  Higher-order interactions are pooled into that error term.

**Language (task B.4).** Report the share, and treat `backbone:adapter` as *corroborating* RQ3,
never as proving it; RQ3's pre-registered matched-budget experiment is the primary evidence. If a
term is small, say so plainly.

In [6]:
EXPECTED_ROWS = {'accuracy': 96, 'ece': 96, 'far_ood_svhn': 96, 'near_ood_tin': 95}

if not RUN_PART_B:
    print('RUN_PART_B = False -- skipped.')
else:
    t0 = time.monotonic()
    RQ1 = M.rq1_interactions(REPO / 'results' / 'mvt_results.json',
                             n_boot=N_BOOT, boot_seed=BOOT_SEED)
    M.dump_json(RQ1, OUT / 'rq1_interactions.json')
    counts = {o: d['n'] for o, d in RQ1['outcomes'].items()}
    pub_ok = {o: d['published_main_effects_check']['ok'] for o, d in RQ1['outcomes'].items()}
    log(f'\n[B] {time.monotonic() - t0:.1f}s | row counts {counts} (expected {EXPECTED_ROWS})')
    log(f'[B] published main effects reproduced: {pub_ok}')
    log(f'[B] OLS vs eta_squared on balanced outcomes: ' + ', '.join(
        f"{o}={d['ols_vs_eta_squared_max_abs_diff']:.1e}"
        for o, d in RQ1['outcomes'].items() if d['balanced']))
    if counts != EXPECTED_ROWS or not all(pub_ok.values()):
        log('\n[B] !! CHECK FAILED -- the row table does not reproduce the published RQ1 table. '
            'Do NOT quote any interaction number from this run. Part A is independent and continues.')
    show(M.rq1_focus_markdown(RQ1, None))
    show(M.rq1_markdown(RQ1, 'RQ1: two-way interactions (committed grid, as published)'))


[B] 6.7s | row counts {'accuracy': 96, 'ece': 96, 'far_ood_svhn': 96, 'near_ood_tin': 95} (expected {'accuracy': 96, 'ece': 96, 'far_ood_svhn': 96, 'near_ood_tin': 95})
[B] published main effects reproduced: {'accuracy': True, 'ece': True, 'far_ood_svhn': True, 'near_ood_tin': True}
[B] OLS vs eta_squared on balanced outcomes: accuracy=1.3e-14, ece=7.6e-16, far_ood_svhn=1.4e-14


| outcome | n | residual (main only) | residual (+2-way) | all 2-way | `backbone:adapter` [95% CI] | largest 2-way term |
|---|---:|---:|---:|---:|---:|---|
| accuracy (as committed) | 96 | 8.95% | 1.31% | 7.64% | 0.73% [0.53%, 0.95%] | dataset:backbone 5.27% |
| ece (as committed) | 96 | 7.70% | 2.05% | 5.65% | 3.28% [2.52%, 4.15%] | backbone:adapter 3.28% |
| far_ood_svhn (as committed) | 96 | 23.87% | 9.60% | 14.27% | 0.18% [0.00%, 0.73%] | dataset:backbone 9.18% |
| near_ood_tin (as committed) | 95 | 11.42% | 6.23% | 5.18% | 0.00% [0.00%, 0.06%] | dataset:backbone 3.81% |

## RQ1: two-way interactions (committed grid, as published)

Factors dataset, k_shot, backbone, adapter, head; baselines excluded (full_ft, linear_probe). Seed bootstrap: 2000 resamples, rng seed 20260914.

### accuracy: `accuracy_mean`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Published main effects reproduced: **True** (max abs diff 0.004 pp). Residual df with two-way terms: 80.

Residual **8.95%** with main effects only → **1.31%** after adding the ten two-way terms (which together take 7.64%).

`backbone:adapter` = **0.73%**, which is 8.15% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| k_shot | 76.05% | [74.90%, 77.39%] | 0.63% | <1e-6 |
| adapter | 9.14% | [8.49%, 9.80%] | 0.34% | <1e-6 |
| dataset:backbone | 5.27% | [4.78%, 5.73%] | 0.25% | <1e-6 |
| backbone | 3.92% | [3.46%, 4.48%] | 0.26% | <1e-6 |
| dataset | 1.75% | [1.46%, 2.05%] | 0.15% | <1e-6 |
| k_shot:backbone | 0.97% | [0.75%, 1.23%] | 0.12% | <1e-6 |
| **backbone:adapter** | 0.73% | [0.53%, 0.95%] | 0.10% | <1e-6 |
| adapter:head | 0.33% | [0.21%, 0.47%] | 0.07% | 2.2e-05 |
| head | 0.19% | [0.10%, 0.29%] | 0.05% | 0.0012 |
| dataset:k_shot | 0.15% | [0.07%, 0.25%] | 0.05% | 0.0036 |
| backbone:head | 0.08% | [0.03%, 0.16%] | 0.04% | 0.028 |
| dataset:adapter | 0.03% | [0.00%, 0.08%] | 0.02% | 0.19 |
| dataset:head | 0.03% | [0.00%, 0.08%] | 0.02% | 0.2 |
| k_shot:head | 0.03% | [0.00%, 0.08%] | 0.02% | 0.21 |
| k_shot:adapter | 0.02% | [0.00%, 0.06%] | 0.02% | 0.3 |
| residual | 1.31% | [1.06%, 1.46%] | 0.10% |  |

### ece: `ece_pooled`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Published main effects reproduced: **True** (max abs diff 0.005 pp). Residual df with two-way terms: 80.

Residual **7.70%** with main effects only → **2.05%** after adding the ten two-way terms (which together take 5.65%).

`backbone:adapter` = **3.28%**, which is 42.60% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| head | 82.89% | [81.97%, 83.87%] | 0.50% | <1e-6 |
| backbone | 4.63% | [3.76%, 5.55%] | 0.46% | <1e-6 |
| **backbone:adapter** | 3.28% | [2.52%, 4.15%] | 0.42% | <1e-6 |
| k_shot | 2.13% | [1.58%, 2.80%] | 0.31% | <1e-6 |
| dataset | 2.02% | [1.42%, 2.70%] | 0.32% | <1e-6 |
| k_shot:backbone | 0.89% | [0.49%, 1.38%] | 0.22% | <1e-6 |
| adapter | 0.63% | [0.32%, 1.04%] | 0.18% | 3.7e-06 |
| dataset:backbone | 0.60% | [0.31%, 0.99%] | 0.18% | 5.9e-06 |
| dataset:adapter | 0.34% | [0.12%, 0.64%] | 0.14% | 0.0005 |
| k_shot:adapter | 0.20% | [0.05%, 0.45%] | 0.11% | 0.0063 |
| k_shot:head | 0.19% | [0.05%, 0.44%] | 0.10% | 0.0073 |
| dataset:head | 0.09% | [0.01%, 0.28%] | 0.07% | 0.065 |
| backbone:head | 0.02% | [0.00%, 0.14%] | 0.04% | 0.36 |
| adapter:head | 0.02% | [0.00%, 0.13%] | 0.04% | 0.41 |
| dataset:k_shot | 0.01% | [0.00%, 0.11%] | 0.03% | 0.48 |
| residual | 2.05% | [1.29%, 2.39%] | 0.28% |  |

### far_ood_svhn: `ood_auroc__svhn_far__{native}`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Published main effects reproduced: **True** (max abs diff 0.004 pp). Residual df with two-way terms: 80.

Residual **23.87%** with main effects only → **9.60%** after adding the ten two-way terms (which together take 14.27%).

`backbone:adapter` = **0.18%**, which is 0.75% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| head | 39.01% | [34.59%, 43.81%] | 2.33% | <1e-6 |
| k_shot | 22.68% | [19.24%, 26.40%] | 1.85% | <1e-6 |
| backbone | 11.54% | [8.93%, 14.75%] | 1.50% | <1e-6 |
| dataset:backbone | 9.18% | [6.68%, 11.79%] | 1.30% | <1e-6 |
| dataset:head | 2.07% | [1.00%, 3.50%] | 0.64% | 8.1e-05 |
| adapter | 1.73% | [0.77%, 2.85%] | 0.54% | 0.00029 |
| backbone:head | 1.35% | [0.55%, 2.49%] | 0.51% | 0.0012 |
| dataset | 1.18% | [0.43%, 2.25%] | 0.47% | 0.0024 |
| k_shot:head | 0.62% | [0.15%, 1.41%] | 0.33% | 0.025 |
| adapter:head | 0.39% | [0.04%, 1.05%] | 0.27% | 0.076 |
| k_shot:backbone | 0.21% | [0.00%, 0.80%] | 0.21% | 0.19 |
| **backbone:adapter** | 0.18% | [0.00%, 0.73%] | 0.20% | 0.22 |
| dataset:k_shot | 0.13% | [0.00%, 0.62%] | 0.17% | 0.31 |
| dataset:adapter | 0.10% | [0.00%, 0.52%] | 0.15% | 0.36 |
| k_shot:adapter | 0.03% | [0.00%, 0.35%] | 0.10% | 0.61 |
| residual | 9.60% | [7.09%, 10.56%] | 0.88% |  |

### near_ood_tin: `ood_auroc__tin_near__{native}`, n=95, balanced=False

Method: OLS Type II (eta_squared is not exact on unbalanced rows). Published main effects reproduced: **True** (max abs diff 0.004 pp). Residual df with two-way terms: 79.

Residual **11.42%** with main effects only → **6.23%** after adding the ten two-way terms (which together take 5.18%).

`backbone:adapter` = **0.00%**, which is 0.00% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| k_shot | 42.86% | [40.41%, 45.00%] | 1.18% | <1e-6 |
| adapter | 21.67% | [20.20%, 23.11%] | 0.73% | <1e-6 |
| head | 21.18% | [19.50%, 23.14%] | 0.94% | <1e-6 |
| dataset:backbone | 3.81% | [3.10%, 4.55%] | 0.37% | <1e-6 |
| dataset | 1.63% | [1.14%, 2.16%] | 0.26% | 2e-05 |
| backbone:head | 0.72% | [0.42%, 1.08%] | 0.17% | 0.0035 |
| backbone | 0.62% | [0.34%, 0.99%] | 0.16% | 0.0062 |
| adapter:head | 0.23% | [0.07%, 0.46%] | 0.10% | 0.095 |
| k_shot:head | 0.16% | [0.04%, 0.35%] | 0.08% | 0.16 |
| dataset:adapter | 0.14% | [0.03%, 0.33%] | 0.08% | 0.18 |
| k_shot:adapter | 0.08% | [0.01%, 0.23%] | 0.06% | 0.33 |
| dataset:k_shot | 0.02% | [0.00%, 0.11%] | 0.03% | 0.63 |
| k_shot:backbone | 0.02% | [0.00%, 0.11%] | 0.03% | 0.64 |
| dataset:head | 0.01% | [0.00%, 0.09%] | 0.03% | 0.71 |
| **backbone:adapter** | 0.00% | [0.00%, 0.06%] | 0.02% | 0.94 |
| residual | 6.23% | [5.06%, 7.19%] | 0.55% |  |

Type-II shares on unbalanced rows do not sum to 100% (here 99.37%).

## 5. Part A: pre-flight + recovery / resume (task A.1)

**Pre-flight (no GPU time).** VAL seeds must be exactly 10000–10099 and TEST seeds 0–599, and the
two must be disjoint. All 21 configs and committed baselines must exist. Baselines that are not
600-episode runs are listed (exactly one is expected). The frozen paths must be clean in this
clone. Any `PROBLEM` line stops the notebook.

**Recovery / resume.** Every attached input is scanned (zips and loose files), and only what these
21 cells need is restored: their checkpoints, their `rq_factorial` records (accepted only as full
600-episode records), plus provenance and logs from an earlier session of this notebook. The scan
is targeted on purpose. `rq_drivers.recover_checkpoints` would stage and copy all ~99 Step 10
checkpoints (~5 GB of disk), and none of them are needed: their scores are already in the
committed `rq_factorial` JSONs.

**Why coverage is counted in records, not with `audit_checkpoints`.** That audit counts `.pt`
files on *this* disk. Here only the 21 are ever present, so it would read ~21/120 by design. The
coverage that matters for RQ2/RQ4 is the number of Phase A records, and Section 9 reports it.

In [7]:
if not RUN_PART_A:
    print('RUN_PART_A = False -- skipped.')
else:
    import rq_core as R
    R.CACHE_OOD_IMAGES = CACHE_OOD_IMAGES

    log('\n=== pre-flight ===')
    PREFLIGHT = M.preflight(REPO, CELLS, log=log)
    M.dump_json(PREFLIGHT, OUT / 'preflight.json')
    if not PREFLIGHT['ok']:
        raise SystemExit('Pre-flight failed -- read the PROBLEM lines above before spending GPU time.')

    log('\n=== recovery / resume ===')
    RECOVERY = M.restore_session_artifacts(REPO, CELLS, search_roots=RECOVER_SEARCH_ROOTS, log=log)
    M.dump_json(RECOVERY, OUT / 'recovery_latest.json')

    COV = M.coverage(REPO, SEL)
    log(f'\n  this session ({COV["n_cells"]} selected cell(s)):')
    log(f'    already evaluated   : {len(COV["evaluated"])}')
    log(f'    checkpoint on disk  : {len(COV["checkpoint_present"])}  (score only, ~4 min each)')
    log(f'    needs training      : {len(COV["needs_training"])}  (measured Step 10 wall time each)')
    log(f'  estimate: ~{COV["estimate_minutes"]:.0f} min ({COV["estimate_minutes"] / 60:.1f} h) '
        f'against MAX_MINUTES={MAX_MINUTES:.0f}')
    if COV['estimate_minutes'] > MAX_MINUTES:
        log('  -> more than this session holds. It stops cleanly at the budget; pack (Section 12), '
            'attach the two datasets next session, and re-run top to bottom.')
    if not RECOVERY['checkpoints'] and COV['needs_training']:
        log('  (no checkpoints recovered -- expected: the 21 are not in any known Step 10 artifact)')


=== pre-flight ===
  VAL seeds 10000..10099 (n=100), TEST seeds 0..599 (n=600), disjoint=True
  21 cell(s): configs + committed baselines present = True
  NOTE baseline is a smoke run: configs/grid/cifar_5shot_mbnet_lora_evidential_seed42.yaml (num_episodes=20) -> first-20-episode guard applies (Section 8)
  frozen paths clean: True

=== recovery / resume ===
  scanned ['/kaggle/input']: 0 zip(s)
  restored: 0 checkpoint(s), 0 factorial record(s), 0 provenance file(s), 0 log(s)

  this session (21 selected cell(s)):
    already evaluated   : 0
    checkpoint on disk  : 0  (score only, ~4 min each)
    needs training      : 21  (measured Step 10 wall time each)
  estimate: ~607 min (10.1 h) against MAX_MINUTES=540
  -> more than this session holds. It stops cleanly at the budget; pack (Section 12), attach the two datasets next session, and re-run top to bottom.
  (no checkpoints recovered -- expected: the 21 are not in any known Step 10 artifact)


## 6. Smoke evaluation (3 episodes on an untrained model, before any training)

This proves the whole scoring path works in *this* session before the first ~20-minute training
starts: the datasets resolve, all four OOD pools build, the VAL-seed tripwire sees 10000..10099,
and `factorial_evaluate` returns every (pool, score) key. The model is untrained, so its numbers
mean nothing, and nothing is written to `rq_factorial/`. It also warms the OOD image cache for the
real run.

In [8]:
if not (RUN_PART_A and RUN_SMOKE_EVAL):
    print('smoke evaluation skipped.')
elif not SEL:
    print('nothing selected.')
else:
    from src.utils import load_config, set_seed
    from src.models import build_model
    from src.evaluators import fit_temperature

    t0 = time.monotonic()
    c0 = SEL[0]
    cfg = load_config(REPO / c0['config'])
    set_seed(int(cfg.seed))
    smoke_model = build_model(cfg).to(DEVICE)     # UNTRAINED: tests the path, not the numbers
    K = int(cfg.dataset.n_way)
    prior = float(cfg.loss.get('prior_per_class', 1.0))
    vl, vt, vseeds = R.load_val_logits(smoke_model, cfg, DEVICE, REPO)
    assert (vseeds[0], vseeds[-1], len(vseeds)) == (10000, 10099, 100), 'VAL seed tripwire'
    T = fit_temperature(vl, vt)
    scale0, bias0 = R.read_evidence_affine(smoke_model.head)
    aff = R.fit_evidence_affine(vl, vt, num_classes=K, prior_per_class=prior,
                                scale_init=scale0, bias_init=bias0)
    pools = R.build_ood_pools(smoke_model, cfg, DEVICE)
    summ = R.factorial_evaluate(smoke_model, cfg, test_seeds=[0, 1, 2], ood_pools=pools,
                                device=DEVICE, temperature=T, affine_valfit=aff,
                                prior_per_class=prior, ece_bins=int(cfg.eval.ece_bins),
                                logits_out=None, log_every=0)
    want_pools = {'svhn_far', 'cifar100_near', 'tin_near', 'gaussian_far'}
    missing = [f'ood_auroc__{p}__{s}' for p in sorted(want_pools) for s in R.FACTORIAL_SCORES
               if f'ood_auroc__{p}__{s}' not in summ]
    log(f'[smoke] pools built: {sorted(pools)}')
    log(f'[smoke] missing (pool, score) keys: {missing or "none"}  [{time.monotonic() - t0:.0f}s]')
    del smoke_model, pools
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if missing:
        raise SystemExit('Smoke evaluation incomplete -- fix data staging (Section 2) before training.')
    log('[smoke] PASSED -- safe to train.')

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 105MB/s]


[smoke] pools built: ['cifar100_near', 'gaussian_far', 'svhn_far', 'tin_near']
[smoke] missing (pool, score) keys: none  [2894s]
[smoke] PASSED -- safe to train.


## 7. Train + score (task A.2)

This is the existing driver, called exactly as the Phase A notebook called it, with
`allow_retrain=True` and the 21 cells. Per cell, `run_phase_a`:
1. trains in-process with the real `scripts/train.py` from the **unchanged committed config**, so
   the recipe and seed are identical to Step 10 (task A.6);
2. fits the temperature and the evidence affine on the **VAL seeds only**, and aborts unless they
   are 10000..10099 and disjoint from the test seeds;
3. scores the 600 frozen test episodes under every (objective, score) combination, and diffs the
   native keys against the committed `results/grid` JSON (the regression guard, read in Section 8).

A cell that errors is logged to `results/rq_factorial/_run_log.jsonl` and the run moves on. Its
checkpoint, if training finished, stays on disk, so re-running only re-scores it.

**Measured Step 10 wall time per cell** (train + one evaluation): mobilenetv3_small/lora/evidential
16–25 min; resnet18/lora 24–30 min; mobilenetv3_small/parallel 23–40 min; resnet18/parallel 24–42
min. Session 1 at `MAX_MINUTES = 540` typically finishes the first three designs and part of the
fourth.

In [9]:
if not RUN_PART_A:
    print('RUN_PART_A = False -- skipped.')
else:
    out_dir = FAC_DIR
    if NUM_EPISODES != M.EXPECTED_N_EPISODES:
        out_dir = OUT / f'_scratch_{NUM_EPISODES}ep'
        log(f'!! NUM_EPISODES={NUM_EPISODES}: writing to {out_dir}, NOT results/rq_factorial/. '
            f'These numbers are not comparable to the grid and never enter the aggregation.')

    had_ckpt = {M.cell_id(c) for c in SEL if (REPO / c['checkpoint']).exists()}
    log(f'\n=== train + score: {len(SEL)} cell(s), budget {MAX_MINUTES:.0f} min ===')
    import rq_drivers as D
    COUNTS = D.run_phase_a(
        REPO, SEL, device=DEVICE, out_dir=out_dir,
        logits_dir=LOGITS_DIR if PERSIST_LOGITS else None,
        num_episodes=NUM_EPISODES, allow_retrain=True,
        wandb_mode=WANDB_MODE, max_minutes=MAX_MINUTES, log=log)
    if out_dir == FAC_DIR:
        M.record_provenance(REPO, SEL, had_ckpt, SESSION_INFO, log=log)

    left = [M.cell_id(c) for c in SEL if not (out_dir / f'{M.cell_id(c)}.json').exists()]
    log(f'\n  counts: {COUNTS}')
    if COUNTS['error']:
        log(f'  {COUNTS["error"]} cell(s) errored -- see results/rq_factorial/_run_log.jsonl '
            f'for each exception (task A.2: investigate individually).')
    if left:
        log(f'  {len(left)} cell(s) not done this session: {left}')
        log('  -> pack (Section 12), attach rq-completion-results + rq-completion-checkpoints '
            'next session, re-run top to bottom. It resumes here.')


=== train + score: 21 cell(s), budget 540 min ===
[A] (1/21) cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed42
    no checkpoint -> training this cell first
[15:42:58] INFO bpeft.train: config: /kaggle/working/thesis/configs/grid/cifar_5shot_mbnet_lora_evidential_seed42.yaml  seed: 42  trainer.type: episodic
[15:42:58] INFO bpeft.train: wandb: disabled (no-op logger)
[15:53:32] INFO bpeft.train: backbone: mobilenetv3_small (feature_dim=576)  adapter: lora/post_pool
[15:53:32] INFO bpeft.train: trainable params: 10,754
[15:54:11] INFO bpeft.train: epoch   1/30  train_loss=0.4140  train_acc=0.837  val_loss=0.5573  val_acc=0.777  kl_w=0.010  mean_ev=1.7596  grad_norm=0.1976  global_step=100
[15:54:48] INFO bpeft.train: epoch   2/30  train_loss=0.3795  train_acc=0.840  val_loss=0.5337  val_acc=0.796  kl_w=0.020  mean_ev=2.1507  grad_norm=0.2331  global_step=200
[15:55:25] INFO bpeft.train: epoch   3/30  train_loss=0.3524  train_acc=0.864  val_loss=0.5249  val_acc=0.803  kl_w=0.030  

100%|██████████| 44.7M/44.7M [00:00<00:00, 181MB/s]


[16:45:16] INFO bpeft.train: backbone: resnet18 (feature_dim=512)  adapter: lora/post_pool
[16:45:16] INFO bpeft.train: trainable params: 12,290
[16:46:02] INFO bpeft.train: epoch   1/30  train_loss=0.5132  train_acc=0.796  val_loss=0.6715  val_acc=0.741  kl_w=0.010  mean_ev=2.9307  grad_norm=0.4594  global_step=100
[16:46:48] INFO bpeft.train: epoch   2/30  train_loss=0.4727  train_acc=0.803  val_loss=0.6706  val_acc=0.728  kl_w=0.020  mean_ev=2.2042  grad_norm=0.4215  global_step=200
[16:47:33] INFO bpeft.train: epoch   3/30  train_loss=0.4414  train_acc=0.821  val_loss=0.6369  val_acc=0.737  kl_w=0.030  mean_ev=2.0152  grad_norm=0.4403  global_step=300
[16:48:19] INFO bpeft.train: epoch   4/30  train_loss=0.4283  train_acc=0.831  val_loss=0.6141  val_acc=0.764  kl_w=0.040  mean_ev=1.9160  grad_norm=0.4675  global_step=400
[16:49:05] INFO bpeft.train: epoch   5/30  train_loss=0.4041  train_acc=0.845  val_loss=0.6047  val_acc=0.745  kl_w=0.050  mean_ev=1.9732  grad_norm=0.4611  global

## 8. Regression guards (task A.4)

**This is the check that matters most.** Each row uses the guard that *applies*:
- **600-episode**, against the committed `results/grid` JSON, for 20 of the 21 cells;
- **first-20-episode**, for `cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed42`, whose
  committed baseline is the smoke run (fact 3). The same checkpoint is re-scored on test seeds 0..19
  with the unchanged `rq_core.regression_guard`. Its 600-episode guard is listed in the JSON as
  `guard_600_episode_status`, and it is expected to read `MISMATCH`.

Two checks that do not depend on evaluation sit alongside:
- **`best_val_epoch`** from the new checkpoint against the committed grid value. For cells trained
  this session, this tests whether *training* reproduced, independently of scoring.
- **`n_params`**, against the committed value.

**How to read it.**
- `exact` / `within_tol` (≤ 1e-6) = reproduced. Expect this; the project's record is 18/18 and 99/99.
- `MISMATCH` together with a different `best_val_epoch` means training diverged. The likely
  cause is the Kaggle image (torch/CUDA/cuDNN) having changed since the Aug 2–6 grid session;
  versions are in `SESSION_INFO`.
- `MISMATCH` with the **same** `best_val_epoch` means the checkpoint probably matches but
  scoring drifted.

Either way it is a **finding to report, not to paper over** (task A.4). The Phase A records stay
internally valid, since each model is scored under all four scores, but the thesis must then say
these 21 are re-trainings, not the Step 10 models.

In [10]:
SMOKE_GUARDS = M.smoke_baseline_guards(REPO, CELLS, device=DEVICE, log=log)
M.dump_json(SMOKE_GUARDS, OUT / 'smoke_baseline_guards.json')
GUARDS = M.guard_table(REPO, CELLS, SMOKE_GUARDS)
M.dump_json(GUARDS, OUT / 'guard_table.json')
show(M.guards_markdown(GUARDS))

mism = [r for r in GUARDS['rows'] if r['guard_status'] == 'MISMATCH']
if mism:
    log(f'\n!! {len(mism)} MISMATCH cell(s). Per task A.4 this is a real finding: record it in the '
        f'report, do not widen a tolerance. best_val_epoch match on those: '
        f'{sum(1 for r in mism if r["best_val_epoch_match"])}/{len(mism)}.')

  cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed42: re-scoring the checkpoint on test seeds 0..19 against the committed 20-episode baseline
    T=0.7295  affine native=(2.8154, -5.9894)  refit=(10.3223, -20.5228)
    guard: exact (8/8 exact, max|diff|=0.00e+00)


### Regression guard against the committed Step 10 grid

Evaluated 21/21. Guard status counts: `exact` 21. Reproduced (exact or within 1e-6): **21/21**. `best_val_epoch` matches: **21/21**. `n_params` all match: True.

| cell | checkpoint | baseline | guard | exact/keys | max abs diff | best_val_epoch committed → new |
|---|---|---|---|---:|---:|---:|
| `cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed42` | trained_this_session | first-20-episode (committed JSON is a smoke run) | `exact` | 8/8 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed43` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 2 → 2 |
| `cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed44` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_resnet18_lora_evidential_seed42` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 4 → 4 |
| `cifar_fs_5shot_resnet18_lora_softmax_seed42` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 6 → 6 |
| `cifar_fs_5shot_resnet18_lora_evidential_seed43` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_resnet18_lora_softmax_seed43` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 4 → 4 |
| `cifar_fs_5shot_resnet18_lora_evidential_seed44` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 7 → 7 |
| `cifar_fs_5shot_resnet18_lora_softmax_seed44` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 4 → 4 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed42` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed42` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 10 → 10 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed43` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed43` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 10 → 10 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed44` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed44` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 14 → 14 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_evidential_seed42` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 19 → 19 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_softmax_seed42` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 10 → 10 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_evidential_seed43` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 2 → 2 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_softmax_seed43` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 2 → 2 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_evidential_seed44` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 12 → 12 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_softmax_seed44` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 10 → 10 |

## 9. RQ2 / RQ4 re-aggregation (task A.5)

Three things happen here, in this order, and they are what make the new numbers quotable:

1. **Aggregator regression.** The unchanged `rq1_verdict` / `rq2_verdict` run on the 99 committed
   records must reproduce `results/rq_summary.json` exactly. If they don't, the environment
   changed the aggregation and no new ratio can be trusted.
2. **Completeness (task A.5.2).** Every design is classified complete / absent / one arm missing /
   seeds missing, and every (design, objective, score) cell with **zero** observations is listed.
   `balanced: True` counts only when the design is also **fully crossed**.
3. **Before / after.** RQ2's far and near score-η², objective-η² and ratio are shown as committed
   (99 records), over all records now on disk, and, when anything is still half-present, over the
   complete designs only. The table says which row is quotable. RQ4's 48/48 and 150/192 headlines
   are shown before and after, plus the new evidential cells alone (task A.5.5).

This is safe on a partial run: it reads whatever records exist.

In [11]:
if not RUN_AGGREGATE:
    print('RUN_AGGREGATE = False -- skipped.')
else:
    AGG = M.rq2_rq4_aggregate(REPO, GRID, NEW_IDS, REPO / 'results' / 'rq_summary.json')
    M.dump_json(AGG, OUT / 'rq2_rq4_summary.json')
    rep = AGG['aggregator_reproduces_committed']
    if not rep['ok']:
        log(f'!! aggregator does NOT reproduce the committed rq_summary.json on the 99 committed '
            f'records ({rep}). Do not quote the new ratio until that is understood.')
    show(M.rq2_rq4_markdown(AGG))

## RQ2 / RQ4 re-aggregation

Phase A factorial records: **120/120** (99 committed + 21 new). Evidential records: 60.

Unchanged aggregator reproduces the committed `results/rq_summary.json` on the committed records: **True** (RQ2 max abs diff 0.0e+00, RQ4 0.0e+00).

Design completeness: 20/20 complete, 0 absent, 0 with one objective arm missing, 0 with seeds missing. Zero-observation objective×score cells: far 0, near 0. **Fully crossed: True.**

### RQ2: objective vs scoring rule (η² share of OOD-AUROC variance)

| variant | pool | score η² | objective η² | score/objective | balanced | n |
|---|---|---:|---:|---:|---|---:|
| before: committed, 99 records | far | 43.68% | 0.267% | 163.4× | True | 792 |
| before: committed, 99 records | near | 12.96% | 0.601% | 21.6× | True | 792 |
| after: all records | far | 42.69% | 0.170% | 250.5× | True | 960 |
| after: all records | near | 14.72% | 0.634% | 23.2× | True | 960 |

Quotable variant: **all records**

Mean AUROC, far-OOD, after (rows = training objective):

| trained as | msp | energy | ts_msp | vacuity_valfit |
|---|---:|---:|---:|---:|
| evidential | 0.7993 (n=120) | 0.9185 (n=120) | 0.7868 (n=120) | 0.9134 (n=120) |
| softmax | 0.8014 (n=120) | 0.9342 (n=120) | 0.7850 (n=120) | 0.9307 (n=120) |

Mean AUROC, near-OOD, after (rows = training objective):

| trained as | msp | energy | ts_msp | vacuity_valfit |
|---|---:|---:|---:|---:|
| evidential | 0.7916 (n=120) | 0.8414 (n=120) | 0.7805 (n=120) | 0.8347 (n=120) |
| softmax | 0.7770 (n=120) | 0.8371 (n=120) | 0.7634 (n=120) | 0.8233 (n=120) |

### RQ4: post-hoc evidence-affine refit (VAL only)

| variant | evidential cells | ECE improved | AUROC preserved (Δ ≥ −0.005) | mean ΔECE | mean ΔAUROC | min Spearman ρ | reordering observed |
|---|---:|---:|---:|---:|---:|---:|---|
| before: committed | 48 | 48/48 | 150/192 | -0.1373 | +0.0040 | 0.9212 | True |
| after: all records | 60 | 60/60 | 192/240 | -0.1387 | +0.0029 | 0.8658 | True |
| new cells only | 12 | 12/12 | 42/48 | -0.1441 | -0.0016 | 0.8658 | True |

## 10. RQ1 sensitivity: the smoke observation replaced

The per-seed RQ1 table contains one observation that is not a 600-episode measurement:
`cifar_fs / 5-shot / mobilenetv3_small / lora / evidential / seed 42` (fact 3). Its accuracy, ECE and
SVHN AUROC are 20-episode estimates, and it has no TinyImageNet value at all. Once Section 7 has
scored that cell **and** its first-20-episode guard in Section 8 is `exact` / `within_tol` (so the
checkpoint on disk is provably the same model), this reruns Part B with that single observation
replaced by the model's 600-episode scores. The replacement keys (`accuracy_mean__evidential_native`,
`ece_pooled__evidential_native`, `ood_auroc__{svhn_far,tin_near}__vacuity_native`) are exactly the
ones the regression guard maps to the grid's keys. TinyImageNet becomes 96 balanced rows.

This is a sensitivity row next to the committed table, not a replacement for it. Report both, and
say which observation changed and why.

In [12]:
if not (RUN_AGGREGATE and RUN_PART_B):
    print('skipped (needs RUN_PART_B and RUN_AGGREGATE).')
else:
    OVR = M.smoke_cell_overrides(REPO, GUARDS)
    if OVR is None:
        log('[B-sens] not available yet: needs the seed-42 cell scored AND its first-20-episode '
            'guard exact/within_tol.')
    else:
        log(f'[B-sens] replacing {list(OVR)} with {OVR}')
        RQ1_CORR = M.rq1_interactions(REPO / 'results' / 'mvt_results.json', overrides=OVR,
                                      n_boot=N_BOOT, boot_seed=BOOT_SEED)
        M.dump_json(RQ1_CORR, OUT / 'rq1_interactions_smoke_obs_replaced.json')
        show(M.rq1_focus_markdown(RQ1, RQ1_CORR))
        show(M.rq1_markdown(RQ1_CORR, 'RQ1: sensitivity (smoke observation replaced)'))

[B-sens] replacing [('cifar_fs', 5, 'mobilenetv3_small', 'lora', 'evidential', 42)] with {('cifar_fs', 5, 'mobilenetv3_small', 'lora', 'evidential', 42): {'accuracy': 0.8757111333807309, 'ece': 0.28426833180586497, 'far_ood_svhn': 0.92873, 'near_ood_tin': 0.8821202222222223}}


| outcome | n | residual (main only) | residual (+2-way) | all 2-way | `backbone:adapter` [95% CI] | largest 2-way term |
|---|---:|---:|---:|---:|---:|---|
| accuracy (as committed) | 96 | 8.95% | 1.31% | 7.64% | 0.73% [0.53%, 0.95%] | dataset:backbone 5.27% |
| ece (as committed) | 96 | 7.70% | 2.05% | 5.65% | 3.28% [2.52%, 4.15%] | backbone:adapter 3.28% |
| far_ood_svhn (as committed) | 96 | 23.87% | 9.60% | 14.27% | 0.18% [0.00%, 0.73%] | dataset:backbone 9.18% |
| near_ood_tin (as committed) | 95 | 11.42% | 6.23% | 5.18% | 0.00% [0.00%, 0.06%] | dataset:backbone 3.81% |
| accuracy (smoke obs. replaced) | 96 | 9.04% | 1.28% | 7.76% | 0.77% [0.58%, 0.98%] | dataset:backbone 5.37% |
| ece (smoke obs. replaced) | 96 | 7.68% | 2.03% | 5.64% | 3.23% [2.49%, 4.05%] | backbone:adapter 3.23% |
| far_ood_svhn (smoke obs. replaced) | 96 | 23.75% | 9.53% | 14.22% | 0.20% [0.00%, 0.75%] | dataset:backbone 9.11% |
| near_ood_tin (smoke obs. replaced) | 96 | 11.49% | 6.21% | 5.28% | 0.00% [0.00%, 0.05%] | dataset:backbone 3.90% |

## RQ1: sensitivity (smoke observation replaced)

Factors dataset, k_shot, backbone, adapter, head; baselines excluded (full_ft, linear_probe). Seed bootstrap: 2000 resamples, rng seed 20260914.
Observations replaced: ['cifar_fs|5|mobilenetv3_small|lora|evidential|42']

### accuracy: `accuracy_mean`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Main effects vs the published table (expected to move): max abs diff 0.26 pp. Residual df with two-way terms: 80.

Residual **9.04%** with main effects only → **1.28%** after adding the ten two-way terms (which together take 7.76%).

`backbone:adapter` = **0.77%**, which is 8.50% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| k_shot | 76.31% | [75.19%, 77.57%] | 0.60% | <1e-6 |
| adapter | 8.98% | [8.35%, 9.60%] | 0.32% | <1e-6 |
| dataset:backbone | 5.37% | [4.88%, 5.82%] | 0.24% | <1e-6 |
| backbone | 3.82% | [3.38%, 4.34%] | 0.24% | <1e-6 |
| dataset | 1.69% | [1.42%, 1.95%] | 0.14% | <1e-6 |
| k_shot:backbone | 1.02% | [0.80%, 1.26%] | 0.12% | <1e-6 |
| **backbone:adapter** | 0.77% | [0.58%, 0.98%] | 0.10% | <1e-6 |
| adapter:head | 0.31% | [0.19%, 0.43%] | 0.06% | 3.8e-05 |
| head | 0.16% | [0.09%, 0.27%] | 0.05% | 0.0019 |
| dataset:k_shot | 0.13% | [0.06%, 0.22%] | 0.04% | 0.0057 |
| backbone:head | 0.10% | [0.04%, 0.18%] | 0.04% | 0.017 |
| dataset:adapter | 0.02% | [0.00%, 0.06%] | 0.02% | 0.26 |
| dataset:head | 0.02% | [0.00%, 0.06%] | 0.02% | 0.27 |
| k_shot:head | 0.02% | [0.00%, 0.06%] | 0.02% | 0.27 |
| k_shot:adapter | 0.01% | [0.00%, 0.05%] | 0.01% | 0.39 |
| residual | 1.28% | [1.04%, 1.43%] | 0.10% |  |

### ece: `ece_pooled`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Main effects vs the published table (expected to move): max abs diff 0.08 pp. Residual df with two-way terms: 80.

Residual **7.68%** with main effects only → **2.03%** after adding the ten two-way terms (which together take 5.64%).

`backbone:adapter` = **3.23%**, which is 42.06% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| head | 82.97% | [82.05%, 83.95%] | 0.49% | <1e-6 |
| backbone | 4.56% | [3.72%, 5.46%] | 0.45% | <1e-6 |
| **backbone:adapter** | 3.23% | [2.49%, 4.05%] | 0.41% | <1e-6 |
| k_shot | 2.16% | [1.63%, 2.82%] | 0.31% | <1e-6 |
| dataset | 1.98% | [1.39%, 2.63%] | 0.32% | <1e-6 |
| k_shot:backbone | 0.91% | [0.52%, 1.40%] | 0.23% | <1e-6 |
| adapter | 0.65% | [0.34%, 1.05%] | 0.18% | 2.6e-06 |
| dataset:backbone | 0.62% | [0.33%, 1.02%] | 0.18% | 4e-06 |
| dataset:adapter | 0.32% | [0.11%, 0.61%] | 0.13% | 0.00063 |
| k_shot:adapter | 0.21% | [0.06%, 0.47%] | 0.11% | 0.0049 |
| k_shot:head | 0.21% | [0.05%, 0.45%] | 0.10% | 0.0057 |
| dataset:head | 0.10% | [0.01%, 0.29%] | 0.07% | 0.054 |
| backbone:head | 0.02% | [0.00%, 0.12%] | 0.03% | 0.41 |
| adapter:head | 0.01% | [0.00%, 0.12%] | 0.03% | 0.45 |
| dataset:k_shot | 0.01% | [0.00%, 0.10%] | 0.03% | 0.53 |
| residual | 2.03% | [1.27%, 2.39%] | 0.28% |  |

### far_ood_svhn: `ood_auroc__svhn_far__{native}`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Main effects vs the published table (expected to move): max abs diff 0.17 pp. Residual df with two-way terms: 80.

Residual **23.75%** with main effects only → **9.53%** after adding the ten two-way terms (which together take 14.22%).

`backbone:adapter` = **0.20%**, which is 0.83% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| head | 38.93% | [34.51%, 43.77%] | 2.33% | <1e-6 |
| k_shot | 22.59% | [19.16%, 26.35%] | 1.84% | <1e-6 |
| backbone | 11.71% | [9.09%, 14.96%] | 1.50% | <1e-6 |
| dataset:backbone | 9.11% | [6.61%, 11.70%] | 1.29% | <1e-6 |
| dataset:head | 2.13% | [1.05%, 3.55%] | 0.65% | 6.1e-05 |
| adapter | 1.79% | [0.82%, 2.92%] | 0.55% | 0.00022 |
| backbone:head | 1.31% | [0.53%, 2.44%] | 0.50% | 0.0014 |
| dataset | 1.23% | [0.47%, 2.33%] | 0.48% | 0.0019 |
| k_shot:head | 0.66% | [0.16%, 1.45%] | 0.34% | 0.021 |
| adapter:head | 0.37% | [0.03%, 1.00%] | 0.26% | 0.083 |
| **backbone:adapter** | 0.20% | [0.00%, 0.75%] | 0.21% | 0.2 |
| k_shot:backbone | 0.19% | [0.00%, 0.77%] | 0.21% | 0.21 |
| dataset:k_shot | 0.11% | [0.00%, 0.61%] | 0.16% | 0.33 |
| dataset:adapter | 0.11% | [0.00%, 0.55%] | 0.16% | 0.33 |
| k_shot:adapter | 0.02% | [0.00%, 0.33%] | 0.09% | 0.65 |
| residual | 9.53% | [7.06%, 10.52%] | 0.88% |  |

### near_ood_tin: `ood_auroc__tin_near__{native}`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Main effects vs the published table (expected to move): max abs diff 0.70 pp. Residual df with two-way terms: 80.

Residual **11.49%** with main effects only → **6.21%** after adding the ten two-way terms (which together take 5.28%).

`backbone:adapter` = **0.00%**, which is 0.00% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| k_shot | 43.30% | [40.94%, 45.56%] | 1.21% | <1e-6 |
| adapter | 21.68% | [20.26%, 23.07%] | 0.73% | <1e-6 |
| head | 21.44% | [19.73%, 23.42%] | 0.91% | <1e-6 |
| dataset:backbone | 3.90% | [3.20%, 4.71%] | 0.38% | <1e-6 |
| dataset | 1.54% | [1.07%, 2.06%] | 0.25% | 2.7e-05 |
| backbone:head | 0.75% | [0.45%, 1.11%] | 0.17% | 0.0026 |
| backbone | 0.55% | [0.29%, 0.89%] | 0.16% | 0.0095 |
| adapter:head | 0.21% | [0.07%, 0.42%] | 0.09% | 0.1 |
| k_shot:head | 0.14% | [0.03%, 0.35%] | 0.08% | 0.18 |
| dataset:adapter | 0.13% | [0.03%, 0.33%] | 0.08% | 0.2 |
| k_shot:adapter | 0.09% | [0.01%, 0.24%] | 0.06% | 0.29 |
| dataset:k_shot | 0.02% | [0.00%, 0.12%] | 0.03% | 0.58 |
| k_shot:backbone | 0.02% | [0.00%, 0.12%] | 0.03% | 0.59 |
| dataset:head | 0.01% | [0.00%, 0.11%] | 0.03% | 0.67 |
| **backbone:adapter** | 0.00% | [0.00%, 0.05%] | 0.01% | 0.98 |
| residual | 6.21% | [5.02%, 7.14%] | 0.55% |  |

## 11. Report + frozen-file check

`results/rq_completion/REPORT.md` collects the four numbers the task asks for (coverage, RQ2
shares and whether the ratio is now stable, RQ4, the interaction tables) along with the guard
table. **Transcribe from this file and the JSONs next to it, never from notebook stdout**, which has
been the repo convention since Step 9.

In [13]:
FROZEN = M.frozen_files_untouched(REPO)
log(f'\nfrozen paths untouched: {FROZEN}')
if FROZEN['ok'] is False:
    log('!! a frozen path changed in this session -- the report says so; do not merge those files back.')
REPORT = M.write_report(OUT / 'REPORT.md', session=SESSION_INFO, frozen=FROZEN, guards=GUARDS,
                        agg=AGG, rq1=RQ1, rq1_corrected=RQ1_CORR)
log(f'wrote {OUT / "REPORT.md"}')
display(Markdown(REPORT))


frozen paths untouched: {'ok': True, 'changed': [], 'paths': ['configs', 'results/grid', 'results/mvt_results.json', 'results/rq_summary.json', 'results/rq_checkpoint_audit.json']}
wrote /kaggle/working/thesis/results/rq_completion/REPORT.md


# RQ2/RQ4 completion + RQ1 interactions: session report

Generated 2026-09-14 22:51:33. GPU `Tesla T4`, torch `2.10.0+cu128`, CUDA `12.8`, repo HEAD `8d706ea`.

Thesis numbering throughout: code `rq1_verdict` = **RQ2**, code `rq2_verdict` = **RQ4**.

Frozen paths untouched (configs, results/grid, results/mvt_results.json, results/rq_summary.json, results/rq_checkpoint_audit.json): **True**

## The four deliverables

1. **Coverage:** 120/120 Phase A records (21/21 new).
2. **RQ2 (all records):** far: score 42.69% / objective 0.170% = 250.5×; near: score 14.72% / objective 0.634% = 23.2×. Fully crossed: True.
3. **RQ4:** ECE improved in 60/60 evidential cells; AUROC preserved in 192/240 comparisons (committed: 48/48 and 150/192).
4. **RQ1 interactions** (four full tables in the RQ1 section below). Headline numbers:

| outcome | n | residual (main only) | residual (+2-way) | all 2-way | `backbone:adapter` [95% CI] | largest 2-way term |
|---|---:|---:|---:|---:|---:|---|
| accuracy (as committed) | 96 | 8.95% | 1.31% | 7.64% | 0.73% [0.53%, 0.95%] | dataset:backbone 5.27% |
| ece (as committed) | 96 | 7.70% | 2.05% | 5.65% | 3.28% [2.52%, 4.15%] | backbone:adapter 3.28% |
| far_ood_svhn (as committed) | 96 | 23.87% | 9.60% | 14.27% | 0.18% [0.00%, 0.73%] | dataset:backbone 9.18% |
| near_ood_tin (as committed) | 95 | 11.42% | 6.23% | 5.18% | 0.00% [0.00%, 0.06%] | dataset:backbone 3.81% |
| accuracy (smoke obs. replaced) | 96 | 9.04% | 1.28% | 7.76% | 0.77% [0.58%, 0.98%] | dataset:backbone 5.37% |
| ece (smoke obs. replaced) | 96 | 7.68% | 2.03% | 5.64% | 3.23% [2.49%, 4.05%] | backbone:adapter 3.23% |
| far_ood_svhn (smoke obs. replaced) | 96 | 23.75% | 9.53% | 14.22% | 0.20% [0.00%, 0.75%] | dataset:backbone 9.11% |
| near_ood_tin (smoke obs. replaced) | 96 | 11.49% | 6.21% | 5.28% | 0.00% [0.00%, 0.05%] | dataset:backbone 3.90% |


### Regression guard against the committed Step 10 grid

Evaluated 21/21. Guard status counts: `exact` 21. Reproduced (exact or within 1e-6): **21/21**. `best_val_epoch` matches: **21/21**. `n_params` all match: True.

| cell | checkpoint | baseline | guard | exact/keys | max abs diff | best_val_epoch committed → new |
|---|---|---|---|---:|---:|---:|
| `cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed42` | trained_this_session | first-20-episode (committed JSON is a smoke run) | `exact` | 8/8 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed43` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 2 → 2 |
| `cifar_fs_5shot_mobilenetv3_small_lora_evidential_seed44` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_resnet18_lora_evidential_seed42` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 4 → 4 |
| `cifar_fs_5shot_resnet18_lora_softmax_seed42` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 6 → 6 |
| `cifar_fs_5shot_resnet18_lora_evidential_seed43` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_resnet18_lora_softmax_seed43` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 4 → 4 |
| `cifar_fs_5shot_resnet18_lora_evidential_seed44` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 7 → 7 |
| `cifar_fs_5shot_resnet18_lora_softmax_seed44` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 4 → 4 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed42` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed42` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 10 → 10 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed43` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed43` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 10 → 10 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_evidential_seed44` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 3 → 3 |
| `cifar_fs_5shot_mobilenetv3_small_bottleneck_parallel_softmax_seed44` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 14 → 14 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_evidential_seed42` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 19 → 19 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_softmax_seed42` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 10 → 10 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_evidential_seed43` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 2 → 2 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_softmax_seed43` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 2 → 2 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_evidential_seed44` | trained_this_session | 600-episode | `exact` | 12/12 | 0.00e+00 | 12 → 12 |
| `cifar_fs_5shot_resnet18_bottleneck_parallel_softmax_seed44` | trained_this_session | 600-episode | `exact` | 13/13 | 0.00e+00 | 10 → 10 |

## RQ2 / RQ4 re-aggregation

Phase A factorial records: **120/120** (99 committed + 21 new). Evidential records: 60.

Unchanged aggregator reproduces the committed `results/rq_summary.json` on the committed records: **True** (RQ2 max abs diff 0.0e+00, RQ4 0.0e+00).

Design completeness: 20/20 complete, 0 absent, 0 with one objective arm missing, 0 with seeds missing. Zero-observation objective×score cells: far 0, near 0. **Fully crossed: True.**

### RQ2: objective vs scoring rule (η² share of OOD-AUROC variance)

| variant | pool | score η² | objective η² | score/objective | balanced | n |
|---|---|---:|---:|---:|---|---:|
| before: committed, 99 records | far | 43.68% | 0.267% | 163.4× | True | 792 |
| before: committed, 99 records | near | 12.96% | 0.601% | 21.6× | True | 792 |
| after: all records | far | 42.69% | 0.170% | 250.5× | True | 960 |
| after: all records | near | 14.72% | 0.634% | 23.2× | True | 960 |

Quotable variant: **all records**

Mean AUROC, far-OOD, after (rows = training objective):

| trained as | msp | energy | ts_msp | vacuity_valfit |
|---|---:|---:|---:|---:|
| evidential | 0.7993 (n=120) | 0.9185 (n=120) | 0.7868 (n=120) | 0.9134 (n=120) |
| softmax | 0.8014 (n=120) | 0.9342 (n=120) | 0.7850 (n=120) | 0.9307 (n=120) |

Mean AUROC, near-OOD, after (rows = training objective):

| trained as | msp | energy | ts_msp | vacuity_valfit |
|---|---:|---:|---:|---:|
| evidential | 0.7916 (n=120) | 0.8414 (n=120) | 0.7805 (n=120) | 0.8347 (n=120) |
| softmax | 0.7770 (n=120) | 0.8371 (n=120) | 0.7634 (n=120) | 0.8233 (n=120) |

### RQ4: post-hoc evidence-affine refit (VAL only)

| variant | evidential cells | ECE improved | AUROC preserved (Δ ≥ −0.005) | mean ΔECE | mean ΔAUROC | min Spearman ρ | reordering observed |
|---|---:|---:|---:|---:|---:|---:|---|
| before: committed | 48 | 48/48 | 150/192 | -0.1373 | +0.0040 | 0.9212 | True |
| after: all records | 60 | 60/60 | 192/240 | -0.1387 | +0.0029 | 0.8658 | True |
| new cells only | 12 | 12/12 | 42/48 | -0.1441 | -0.0016 | 0.8658 | True |

## RQ1: two-way interactions (committed grid, as published)

Factors dataset, k_shot, backbone, adapter, head; baselines excluded (full_ft, linear_probe). Seed bootstrap: 2000 resamples, rng seed 20260914.

### accuracy: `accuracy_mean`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Published main effects reproduced: **True** (max abs diff 0.004 pp). Residual df with two-way terms: 80.

Residual **8.95%** with main effects only → **1.31%** after adding the ten two-way terms (which together take 7.64%).

`backbone:adapter` = **0.73%**, which is 8.15% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| k_shot | 76.05% | [74.90%, 77.39%] | 0.63% | <1e-6 |
| adapter | 9.14% | [8.49%, 9.80%] | 0.34% | <1e-6 |
| dataset:backbone | 5.27% | [4.78%, 5.73%] | 0.25% | <1e-6 |
| backbone | 3.92% | [3.46%, 4.48%] | 0.26% | <1e-6 |
| dataset | 1.75% | [1.46%, 2.05%] | 0.15% | <1e-6 |
| k_shot:backbone | 0.97% | [0.75%, 1.23%] | 0.12% | <1e-6 |
| **backbone:adapter** | 0.73% | [0.53%, 0.95%] | 0.10% | <1e-6 |
| adapter:head | 0.33% | [0.21%, 0.47%] | 0.07% | 2.2e-05 |
| head | 0.19% | [0.10%, 0.29%] | 0.05% | 0.0012 |
| dataset:k_shot | 0.15% | [0.07%, 0.25%] | 0.05% | 0.0036 |
| backbone:head | 0.08% | [0.03%, 0.16%] | 0.04% | 0.028 |
| dataset:adapter | 0.03% | [0.00%, 0.08%] | 0.02% | 0.19 |
| dataset:head | 0.03% | [0.00%, 0.08%] | 0.02% | 0.2 |
| k_shot:head | 0.03% | [0.00%, 0.08%] | 0.02% | 0.21 |
| k_shot:adapter | 0.02% | [0.00%, 0.06%] | 0.02% | 0.3 |
| residual | 1.31% | [1.06%, 1.46%] | 0.10% |  |

### ece: `ece_pooled`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Published main effects reproduced: **True** (max abs diff 0.005 pp). Residual df with two-way terms: 80.

Residual **7.70%** with main effects only → **2.05%** after adding the ten two-way terms (which together take 5.65%).

`backbone:adapter` = **3.28%**, which is 42.60% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| head | 82.89% | [81.97%, 83.87%] | 0.50% | <1e-6 |
| backbone | 4.63% | [3.76%, 5.55%] | 0.46% | <1e-6 |
| **backbone:adapter** | 3.28% | [2.52%, 4.15%] | 0.42% | <1e-6 |
| k_shot | 2.13% | [1.58%, 2.80%] | 0.31% | <1e-6 |
| dataset | 2.02% | [1.42%, 2.70%] | 0.32% | <1e-6 |
| k_shot:backbone | 0.89% | [0.49%, 1.38%] | 0.22% | <1e-6 |
| adapter | 0.63% | [0.32%, 1.04%] | 0.18% | 3.7e-06 |
| dataset:backbone | 0.60% | [0.31%, 0.99%] | 0.18% | 5.9e-06 |
| dataset:adapter | 0.34% | [0.12%, 0.64%] | 0.14% | 0.0005 |
| k_shot:adapter | 0.20% | [0.05%, 0.45%] | 0.11% | 0.0063 |
| k_shot:head | 0.19% | [0.05%, 0.44%] | 0.10% | 0.0073 |
| dataset:head | 0.09% | [0.01%, 0.28%] | 0.07% | 0.065 |
| backbone:head | 0.02% | [0.00%, 0.14%] | 0.04% | 0.36 |
| adapter:head | 0.02% | [0.00%, 0.13%] | 0.04% | 0.41 |
| dataset:k_shot | 0.01% | [0.00%, 0.11%] | 0.03% | 0.48 |
| residual | 2.05% | [1.29%, 2.39%] | 0.28% |  |

### far_ood_svhn: `ood_auroc__svhn_far__{native}`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Published main effects reproduced: **True** (max abs diff 0.004 pp). Residual df with two-way terms: 80.

Residual **23.87%** with main effects only → **9.60%** after adding the ten two-way terms (which together take 14.27%).

`backbone:adapter` = **0.18%**, which is 0.75% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| head | 39.01% | [34.59%, 43.81%] | 2.33% | <1e-6 |
| k_shot | 22.68% | [19.24%, 26.40%] | 1.85% | <1e-6 |
| backbone | 11.54% | [8.93%, 14.75%] | 1.50% | <1e-6 |
| dataset:backbone | 9.18% | [6.68%, 11.79%] | 1.30% | <1e-6 |
| dataset:head | 2.07% | [1.00%, 3.50%] | 0.64% | 8.1e-05 |
| adapter | 1.73% | [0.77%, 2.85%] | 0.54% | 0.00029 |
| backbone:head | 1.35% | [0.55%, 2.49%] | 0.51% | 0.0012 |
| dataset | 1.18% | [0.43%, 2.25%] | 0.47% | 0.0024 |
| k_shot:head | 0.62% | [0.15%, 1.41%] | 0.33% | 0.025 |
| adapter:head | 0.39% | [0.04%, 1.05%] | 0.27% | 0.076 |
| k_shot:backbone | 0.21% | [0.00%, 0.80%] | 0.21% | 0.19 |
| **backbone:adapter** | 0.18% | [0.00%, 0.73%] | 0.20% | 0.22 |
| dataset:k_shot | 0.13% | [0.00%, 0.62%] | 0.17% | 0.31 |
| dataset:adapter | 0.10% | [0.00%, 0.52%] | 0.15% | 0.36 |
| k_shot:adapter | 0.03% | [0.00%, 0.35%] | 0.10% | 0.61 |
| residual | 9.60% | [7.09%, 10.56%] | 0.88% |  |

### near_ood_tin: `ood_auroc__tin_near__{native}`, n=95, balanced=False

Method: OLS Type II (eta_squared is not exact on unbalanced rows). Published main effects reproduced: **True** (max abs diff 0.004 pp). Residual df with two-way terms: 79.

Residual **11.42%** with main effects only → **6.23%** after adding the ten two-way terms (which together take 5.18%).

`backbone:adapter` = **0.00%**, which is 0.00% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| k_shot | 42.86% | [40.41%, 45.00%] | 1.18% | <1e-6 |
| adapter | 21.67% | [20.20%, 23.11%] | 0.73% | <1e-6 |
| head | 21.18% | [19.50%, 23.14%] | 0.94% | <1e-6 |
| dataset:backbone | 3.81% | [3.10%, 4.55%] | 0.37% | <1e-6 |
| dataset | 1.63% | [1.14%, 2.16%] | 0.26% | 2e-05 |
| backbone:head | 0.72% | [0.42%, 1.08%] | 0.17% | 0.0035 |
| backbone | 0.62% | [0.34%, 0.99%] | 0.16% | 0.0062 |
| adapter:head | 0.23% | [0.07%, 0.46%] | 0.10% | 0.095 |
| k_shot:head | 0.16% | [0.04%, 0.35%] | 0.08% | 0.16 |
| dataset:adapter | 0.14% | [0.03%, 0.33%] | 0.08% | 0.18 |
| k_shot:adapter | 0.08% | [0.01%, 0.23%] | 0.06% | 0.33 |
| dataset:k_shot | 0.02% | [0.00%, 0.11%] | 0.03% | 0.63 |
| k_shot:backbone | 0.02% | [0.00%, 0.11%] | 0.03% | 0.64 |
| dataset:head | 0.01% | [0.00%, 0.09%] | 0.03% | 0.71 |
| **backbone:adapter** | 0.00% | [0.00%, 0.06%] | 0.02% | 0.94 |
| residual | 6.23% | [5.06%, 7.19%] | 0.55% |  |

Type-II shares on unbalanced rows do not sum to 100% (here 99.37%).

## RQ1: sensitivity (the 20-episode smoke observation replaced by the same model's 600-episode scores)

Factors dataset, k_shot, backbone, adapter, head; baselines excluded (full_ft, linear_probe). Seed bootstrap: 2000 resamples, rng seed 20260914.
Observations replaced: ['cifar_fs|5|mobilenetv3_small|lora|evidential|42']

### accuracy: `accuracy_mean`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Main effects vs the published table (expected to move): max abs diff 0.26 pp. Residual df with two-way terms: 80.

Residual **9.04%** with main effects only → **1.28%** after adding the ten two-way terms (which together take 7.76%).

`backbone:adapter` = **0.77%**, which is 8.50% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| k_shot | 76.31% | [75.19%, 77.57%] | 0.60% | <1e-6 |
| adapter | 8.98% | [8.35%, 9.60%] | 0.32% | <1e-6 |
| dataset:backbone | 5.37% | [4.88%, 5.82%] | 0.24% | <1e-6 |
| backbone | 3.82% | [3.38%, 4.34%] | 0.24% | <1e-6 |
| dataset | 1.69% | [1.42%, 1.95%] | 0.14% | <1e-6 |
| k_shot:backbone | 1.02% | [0.80%, 1.26%] | 0.12% | <1e-6 |
| **backbone:adapter** | 0.77% | [0.58%, 0.98%] | 0.10% | <1e-6 |
| adapter:head | 0.31% | [0.19%, 0.43%] | 0.06% | 3.8e-05 |
| head | 0.16% | [0.09%, 0.27%] | 0.05% | 0.0019 |
| dataset:k_shot | 0.13% | [0.06%, 0.22%] | 0.04% | 0.0057 |
| backbone:head | 0.10% | [0.04%, 0.18%] | 0.04% | 0.017 |
| dataset:adapter | 0.02% | [0.00%, 0.06%] | 0.02% | 0.26 |
| dataset:head | 0.02% | [0.00%, 0.06%] | 0.02% | 0.27 |
| k_shot:head | 0.02% | [0.00%, 0.06%] | 0.02% | 0.27 |
| k_shot:adapter | 0.01% | [0.00%, 0.05%] | 0.01% | 0.39 |
| residual | 1.28% | [1.04%, 1.43%] | 0.10% |  |

### ece: `ece_pooled`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Main effects vs the published table (expected to move): max abs diff 0.08 pp. Residual df with two-way terms: 80.

Residual **7.68%** with main effects only → **2.03%** after adding the ten two-way terms (which together take 5.64%).

`backbone:adapter` = **3.23%**, which is 42.06% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| head | 82.97% | [82.05%, 83.95%] | 0.49% | <1e-6 |
| backbone | 4.56% | [3.72%, 5.46%] | 0.45% | <1e-6 |
| **backbone:adapter** | 3.23% | [2.49%, 4.05%] | 0.41% | <1e-6 |
| k_shot | 2.16% | [1.63%, 2.82%] | 0.31% | <1e-6 |
| dataset | 1.98% | [1.39%, 2.63%] | 0.32% | <1e-6 |
| k_shot:backbone | 0.91% | [0.52%, 1.40%] | 0.23% | <1e-6 |
| adapter | 0.65% | [0.34%, 1.05%] | 0.18% | 2.6e-06 |
| dataset:backbone | 0.62% | [0.33%, 1.02%] | 0.18% | 4e-06 |
| dataset:adapter | 0.32% | [0.11%, 0.61%] | 0.13% | 0.00063 |
| k_shot:adapter | 0.21% | [0.06%, 0.47%] | 0.11% | 0.0049 |
| k_shot:head | 0.21% | [0.05%, 0.45%] | 0.10% | 0.0057 |
| dataset:head | 0.10% | [0.01%, 0.29%] | 0.07% | 0.054 |
| backbone:head | 0.02% | [0.00%, 0.12%] | 0.03% | 0.41 |
| adapter:head | 0.01% | [0.00%, 0.12%] | 0.03% | 0.45 |
| dataset:k_shot | 0.01% | [0.00%, 0.10%] | 0.03% | 0.53 |
| residual | 2.03% | [1.27%, 2.39%] | 0.28% |  |

### far_ood_svhn: `ood_auroc__svhn_far__{native}`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Main effects vs the published table (expected to move): max abs diff 0.17 pp. Residual df with two-way terms: 80.

Residual **23.75%** with main effects only → **9.53%** after adding the ten two-way terms (which together take 14.22%).

`backbone:adapter` = **0.20%**, which is 0.83% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| head | 38.93% | [34.51%, 43.77%] | 2.33% | <1e-6 |
| k_shot | 22.59% | [19.16%, 26.35%] | 1.84% | <1e-6 |
| backbone | 11.71% | [9.09%, 14.96%] | 1.50% | <1e-6 |
| dataset:backbone | 9.11% | [6.61%, 11.70%] | 1.29% | <1e-6 |
| dataset:head | 2.13% | [1.05%, 3.55%] | 0.65% | 6.1e-05 |
| adapter | 1.79% | [0.82%, 2.92%] | 0.55% | 0.00022 |
| backbone:head | 1.31% | [0.53%, 2.44%] | 0.50% | 0.0014 |
| dataset | 1.23% | [0.47%, 2.33%] | 0.48% | 0.0019 |
| k_shot:head | 0.66% | [0.16%, 1.45%] | 0.34% | 0.021 |
| adapter:head | 0.37% | [0.03%, 1.00%] | 0.26% | 0.083 |
| **backbone:adapter** | 0.20% | [0.00%, 0.75%] | 0.21% | 0.2 |
| k_shot:backbone | 0.19% | [0.00%, 0.77%] | 0.21% | 0.21 |
| dataset:k_shot | 0.11% | [0.00%, 0.61%] | 0.16% | 0.33 |
| dataset:adapter | 0.11% | [0.00%, 0.55%] | 0.16% | 0.33 |
| k_shot:adapter | 0.02% | [0.00%, 0.33%] | 0.09% | 0.65 |
| residual | 9.53% | [7.06%, 10.52%] | 0.88% |  |

### near_ood_tin: `ood_auroc__tin_near__{native}`, n=96, balanced=True

Method: eta_squared (exact: balanced design). Main effects vs the published table (expected to move): max abs diff 0.70 pp. Residual df with two-way terms: 80.

Residual **11.49%** with main effects only → **6.21%** after adding the ten two-way terms (which together take 5.28%).

`backbone:adapter` = **0.00%**, which is 0.00% of the main-effects residual.

| term | share | 95% seed bootstrap | bootstrap SD | F-test p |
|---|---:|---:|---:|---:|
| k_shot | 43.30% | [40.94%, 45.56%] | 1.21% | <1e-6 |
| adapter | 21.68% | [20.26%, 23.07%] | 0.73% | <1e-6 |
| head | 21.44% | [19.73%, 23.42%] | 0.91% | <1e-6 |
| dataset:backbone | 3.90% | [3.20%, 4.71%] | 0.38% | <1e-6 |
| dataset | 1.54% | [1.07%, 2.06%] | 0.25% | 2.7e-05 |
| backbone:head | 0.75% | [0.45%, 1.11%] | 0.17% | 0.0026 |
| backbone | 0.55% | [0.29%, 0.89%] | 0.16% | 0.0095 |
| adapter:head | 0.21% | [0.07%, 0.42%] | 0.09% | 0.1 |
| k_shot:head | 0.14% | [0.03%, 0.35%] | 0.08% | 0.18 |
| dataset:adapter | 0.13% | [0.03%, 0.33%] | 0.08% | 0.2 |
| k_shot:adapter | 0.09% | [0.01%, 0.24%] | 0.06% | 0.29 |
| dataset:k_shot | 0.02% | [0.00%, 0.12%] | 0.03% | 0.58 |
| k_shot:backbone | 0.02% | [0.00%, 0.12%] | 0.03% | 0.59 |
| dataset:head | 0.01% | [0.00%, 0.11%] | 0.03% | 0.67 |
| **backbone:adapter** | 0.00% | [0.00%, 0.05%] | 0.01% | 0.98 |
| residual | 6.21% | [5.02%, 7.14%] | 0.55% |  |


## 12. Pack + push artifacts

Three zips, so a failed push of a large one cannot take the small one with it:

| zip | contents | size | why |
|---|---|---|---|
| `rq_completion_results.zip` | the new `rq_factorial/<cell>.json` records, `_run_log.jsonl`, all of `results/rq_completion/`, `scripts/rq_completion.py` | < 5 MB | what gets merged into the repo |
| `rq_completion_checkpoints.zip` | the 21 cells' `.pt` files | ~600 MB | lets the next session resume, and makes sure these checkpoints never go missing again |
| `rq_completion_logits.zip` | per-episode logits for the 21 cells | ~250 MB | future re-scoring without a GPU |

Paths inside each zip are repo-relative, so Section 5 restores them next session and a local
`unzip` at the repo root puts them in place. With `KAGGLE_USERNAME` / `KAGGLE_KEY` Kaggle Secrets
the zips are pushed as private datasets. Otherwise, download links are shown.

In [14]:
import glob, hashlib, zipfile

def _pack(zip_path, files, stem):
    files = [p for p in files if os.path.isfile(p)]
    if not files:
        print(f'  {stem}: nothing to pack'); return None
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in files:
            zf.write(p)
        zf.writestr('MANIFEST.txt', '\n'.join(
            f'{p}  {os.path.getsize(p)}B  '
            f'sha256={hashlib.sha256(open(p, "rb").read()).hexdigest()}'
            for p in files))
    mb = os.path.getsize(zip_path) / 1e6
    print(f'  wrote {zip_path} ({len(files)} files, {mb:.1f} MB)')
    return zip_path

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else str(REPO.parent)
ids = sorted(NEW_IDS)

results_files = (
    [f'results/rq_factorial/{i}.json' for i in ids]
    + ['results/rq_factorial/_run_log.jsonl']
    + sorted(p for p in glob.glob(f'results/{M.OUT_SUBDIR}/**/*', recursive=True)
             if '/_scratch_' not in p)
    + ['scripts/rq_completion.py']
)
zip_results = _pack(f'{WORK}/rq_completion_results.zip', results_files, 'results')
zip_ckpts = (_pack(f'{WORK}/rq_completion_checkpoints.zip', [c['checkpoint'] for c in CELLS],
                   'checkpoints') if PACK_CHECKPOINTS else None)
zip_logits = (_pack(f'{WORK}/rq_completion_logits.zip',
                    [f'results/rq_logits/{i}.npz' for i in ids], 'logits')
              if PERSIST_LOGITS else None)

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = _s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = _s.get_secret('KAGGLE_KEY')
    HAVE_SECRETS = True
except Exception:
    HAVE_SECRETS = False

for zp, stem in ((zip_results, 'rq-completion-results'),
                 (zip_ckpts, 'rq-completion-checkpoints'),
                 (zip_logits, 'rq-completion-logits')):
    if zp is None:
        continue
    if HAVE_SECRETS:
        ds_dir = f'{WORK}/{stem}_dataset'
        os.makedirs(ds_dir, exist_ok=True)
        subprocess.run(['cp', zp, ds_dir], check=True)
        json.dump({'title': stem,
                   'id': f"{os.environ['KAGGLE_USERNAME']}/{stem}",
                   'licenses': [{'name': 'CC0-1.0'}]},
                  open(f'{ds_dir}/dataset-metadata.json', 'w'))
        r = subprocess.run(['kaggle', 'datasets', 'create', '-p', ds_dir, '-q'],
                           capture_output=True, text=True)
        if r.returncode != 0:
            subprocess.run(['kaggle', 'datasets', 'version', '-p', ds_dir, '-m', 'update', '-q'])
        print(f'  pushed Kaggle dataset: {stem}')
    else:
        from IPython.display import FileLink
        display(FileLink(zp))
print('\nNo Kaggle Secrets -> use the download links above (or the Output tab of the saved version).'
      if not HAVE_SECRETS else '\nPushed. These survive the tab closing.')

  wrote /kaggle/working/rq_completion_results.zip (54 files, 0.1 MB)
  wrote /kaggle/working/rq_completion_checkpoints.zip (21 files, 531.5 MB)
  wrote /kaggle/working/rq_completion_logits.zip (21 files, 229.5 MB)
  pushed Kaggle dataset: rq-completion-results
  pushed Kaggle dataset: rq-completion-checkpoints
  pushed Kaggle dataset: rq-completion-logits

Pushed. These survive the tab closing.


## After this session

1. **Open `REPORT.md` and read the guard table first.** Every evaluated cell should be `exact` or
   `within_tol`. Anything else goes in the write-up as a finding (Section 8 says how to read it).
   Do not quote "byte-identical reproducibility" for this slice until it is clean.
2. **If cells are left** (expected after session 1), attach `rq-completion-results` **and**
   `rq-completion-checkpoints` (plus `beft-thesis-data`) and re-run top to bottom. Section 5
   restores both and training resumes at the next cell; Part B and the aggregation recompute.
3. **When all 21 are in and the report says `Fully crossed: True`**, unzip
   `rq_completion_results.zip` at the repo root locally. That adds the 21 `results/rq_factorial/*.json`
   records, `results/rq_completion/`, and `scripts/rq_completion.py`. **Do not** add the checkpoint
   or logits zips to git (task A.6). A human reviews and commits.
4. **Update the docs from `REPORT.md`** (task A.5.4–5, B.4):
   - `docs/DEFENCE_DECK_25_SLIDES.md` Slide 20: replace the "99 of 120" coverage box with the new
     coverage and the quotable RQ2 ratio, and remove the "163×/394×" note now that its cause is
     fixed. Slide 24: limitation 8. Slide 22: only if RQ4's headline moved. Slide 19: the
     "stated limitation" box becomes the interaction result, worded as corroboration of RQ3.
   - `docs/RQ_SUPERVISOR_REPORT.md` and `docs/RQ_RESULTS_SUMMARY.md` (RQ1, RQ2, RQ4 sections).
   - Replace the RQ1 "data-quality note" (95/96 TinyImageNet) with the actual cause: one Step 10a
     smoke-test observation (20 episodes) that `--resume` kept. Say whether the sensitivity row in
     Section 10 moves any share. Log it in `progress.txt`'s decisions log.
5. Keep RQ3's order of evidence: the matched-budget experiment is primary, and the
   `backbone:adapter` term is corroborating.